**SCRIPT 1:** *The Historical Data Engine*

In [ ]:
# ==============================================================================
# SCRIPT 1: The Historical Data Engine
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install nba_api pandas unidecode
!apt-get update --quiet
!apt-get install -y chromium-chromedriver --quiet

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import re
import time
import os
from datetime import datetime
from unidecode import unidecode
from io import StringIO
from google.colab import drive
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelogs, teamestimatedmetrics
from IPython.display import display

# ==============================================================================
# Part 2: Helper Functions
# ==============================================================================

def normalize_player_name(name):
    """Cleans and standardizes a player's name for reliable matching."""
    if not isinstance(name, str): return None
    name = unidecode(name.lower())
    name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name)
    name = re.sub(r'[^\w\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

def fetch_team_metrics(season):
    """Fetches advanced team metrics for a given season."""
    print(f"Fetching team metrics for season: {season}...")
    try:
        metrics = teamestimatedmetrics.TeamEstimatedMetrics(season=season).get_data_frames()[0]
        team_map = {team['id']: team['abbreviation'] for team in teams.get_teams()}
        metrics['TEAM_ABBREVIATION'] = metrics['TEAM_ID'].map(team_map)
        print("Successfully fetched team metrics.")
        return metrics
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return pd.DataFrame()

# ==============================================================================
# Part 3: The Historical Data Generation Engine
# ==============================================================================

def build_historical_dataset(season_string):
    """
    Main function to build a complete training dataset for a single season,
    with all requested features and target variables.
    """
    # --- MOUNT DRIVE AND SET UP PATHS ---
    try:
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        OUTPUT_DIR = "/content/drive/MyDrive/NBA/Historical Data"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"Output directory set to: {OUTPUT_DIR}")
    except Exception as e:
        print(f"Could not mount Google Drive. Aborting. Error: {e}")
        return

    output_filename = f"nba_training_data_{season_string}.csv"
    full_output_path = os.path.join(OUTPUT_DIR, output_filename)

    # --- CONFIRM OVERWRITE ---
    if os.path.exists(full_output_path):
        confirm = input(f"WARNING: '{output_filename}' already exists in your Google Drive. Overwrite? (y/n): ").lower()
        if confirm != 'y':
            print("Aborting script.")
            return
        print("Proceeding with overwrite...")

    print(f"\n--- Starting Historical Data Build for {season_string} ---")

    team_metrics_df = fetch_team_metrics(season=season_string)

    print(f"Fetching all player game logs for {season_string}...")
    try:
        # PlayerGameLogs is the most efficient endpoint for this task
        all_gamelogs = playergamelogs.PlayerGameLogs(season_nullable=season_string).get_data_frames()[0]
    except Exception as e:
        print(f"Could not fetch game logs. Aborting. Error: {e}")
        return

    # --- FILTER FOR SIGNIFICANT MINUTES ---
    MINIMUM_MINUTES_THRESHOLD = 15
    original_rows = len(all_gamelogs)
    df = all_gamelogs[all_gamelogs['MIN'] >= MINIMUM_MINUTES_THRESHOLD].copy()
    filtered_rows = len(df)
    print(f"Filtered for significant playing time (>= {MINIMUM_MINUTES_THRESHOLD} mins). Kept {filtered_rows} of {original_rows} game entries.")

    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df.sort_values(by=['PLAYER_ID', 'GAME_DATE'], inplace=True)

    print("Engineering features: Rolling Averages and Days of Rest...")
    stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']
    windows = [3, 5, 10]
    grouped = df.groupby('PLAYER_ID')

    for stat in stats_to_roll:
        for w in windows:
            # Use min_periods=1 to generate averages even with fewer than `w` games
            df[f'ROLL_AVG_{stat}_{w}'] = grouped[stat].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())

    df['DAYS_OF_REST'] = grouped['GAME_DATE'].transform(lambda x: x.diff().dt.days - 1)
    df.loc[:, 'DAYS_OF_REST'] = df['DAYS_OF_REST'].fillna(7) # Assume 1 week rest for first game of season

    print("Engineering features: Opponent Metrics...")
    df['OPPONENT_ABBR'] = df['MATCHUP'].apply(lambda x: x[-3:] if isinstance(x, str) else None)

    team_metrics_df_opp = team_metrics_df.add_suffix('_OPP')
    df = pd.merge(df, team_metrics_df_opp, left_on='OPPONENT_ABBR', right_on='TEAM_ABBREVIATION_OPP', how='left')

    # --- DEFINE ALL TARGET VARIABLES ---
    print("Defining all target variables...")
    # Regression Targets
    df['TARGET_PTS'] = df['PTS']
    df['TARGET_REB'] = df['REB']
    df['TARGET_AST'] = df['AST']
    df['TARGET_P+R'] = df['PTS'] + df['REB']
    df['TARGET_P+A'] = df['PTS'] + df['AST']
    df['TARGET_R+A'] = df['REB'] + df['AST']
    df['TARGET_PRA'] = df['PTS'] + df['REB'] + df['AST']

    # Classification Targets
    stat_columns = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    df['DD_COUNT'] = df[stat_columns].ge(10).sum(axis=1)
    df['TARGET_DD'] = (df['DD_COUNT'] >= 2).astype(int)
    df['TARGET_TD'] = (df['DD_COUNT'] >= 3).astype(int)

    # Drop rows where rolling averages or opponent data could not be calculated
    df.dropna(inplace=True)

    feature_columns = [
        'ROLL_AVG_PTS_3', 'ROLL_AVG_REB_3', 'ROLL_AVG_AST_3',
        'ROLL_AVG_PTS_5', 'ROLL_AVG_REB_5', 'ROLL_AVG_AST_5',
        'ROLL_AVG_PTS_10', 'ROLL_AVG_REB_10', 'ROLL_AVG_AST_10',
        'ROLL_AVG_MIN_10', 'ROLL_AVG_FGA_10', 'ROLL_AVG_FTA_10', 'ROLL_AVG_FG3A_10',
        'DAYS_OF_REST',
        'E_PACE_OPP', 'E_DEF_RATING_OPP'
    ]
    target_columns = [
        'TARGET_PTS', 'TARGET_REB', 'TARGET_AST',
        'TARGET_P+R', 'TARGET_P+A', 'TARGET_R+A', 'TARGET_PRA',
        'TARGET_DD', 'TARGET_TD'
    ]

    final_df = df[feature_columns + target_columns]

    final_df.to_csv(full_output_path, index=False)

    print(f"\n✅ Historical dataset build complete.")
    print(f"Generated {len(final_df)} training rows.")
    print(f"File saved to: {full_output_path}")

    display(final_df.head())

# ==============================================================================
# Part 5: Run the script
# ==============================================================================
if __name__ == "__main__":
    # Define the historical season you want to build the training data for.
    # We use the last completed season for the most data.
    SEASON_TO_BUILD = "2024-25"
    build_historical_dataset(season_string=SEASON_TO_BUILD)

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,594 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,798 kB]
Fetched 5,649 kB in 3s (1,801 kB/s)
Readi

,ROLL_AVG_PTS_3,ROLL_AVG_REB_3,ROLL_AVG_AST_3,ROLL_AVG_PTS_5,ROLL_AVG_REB_5,ROLL_AVG_AST_5,ROLL_AVG_PTS_10,ROLL_AVG_REB_10,ROLL_AVG_AST_10,ROLL_AVG_MIN_10,...,E_DEF_RATING_OPP,TARGET_PTS,TARGET_REB,TARGET_AST,TARGET_P+R,TARGET_P+A,TARGET_R+A,TARGET_PRA,TARGET_DD,TARGET_TD
1,19.000000,5.000000,4.000000,19.0,5.000000,4.000000,19.0,5.000000,4.000000,16.333333,...,111.2,11,6,3,17,14,9,20,0,0
2,15.000000,5.500000,3.500000,15.0,5.500000,3.500000,15.0,5.500000,3.500000,16.075000,...,108.7,6,2,4,8,10,6,12,0,0
3,12.000000,4.333333,3.666667,12.0,4.333333,3.666667,12.0,4.333333,3.666667,18.527778,...,108.7,16,5,4,21,20,9,25,0,0
4,11.000000,4.333333,3.666667,13.0,4.500000,3.750000,13.0,4.500000,3.750000,22.558333,...,116.7,21,4,8,25,29,12,33,0,0
5,14.333333,3.666667,5.333333,14.6,4.400000,4.600000,14.6,4.400000,4.600000,24.986667,...,113.3,32,14,10,46,42,24,56,1,1


**SCRIPT 2:** *train_model.py*

In [ ]:
# ==============================================================================
# SCRIPT 2, PART 1: INSTALLATION
# ==============================================================================
# This cell's only job is to set up the correct library versions.

!pip install scikit-learn==1.3.0 pandas joblib numpy==1.26.4

print("\n" + "="*60)
print("--> IMPORTANT: INSTALLATION COMPLETE. <--" )
print("--> You MUST restart the runtime now before running the next cell. <--" )
print("--> Go to the 'Runtime' menu above and select 'Restart session'. <--" )
print("="*60)

# This line will automatically crash the kernel to force a restart.
# After it crashes, you are ready to run the second cell.
import os
os.kill(os.getpid(), 9)

### **SCRIPT 2** - *PART 2: MODEL TRAINING*

In [ ]:
# ==============================================================================
# SCRIPT 2, PART 2: MODEL TRAINING
# ==============================================================================
# Run this cell ONLY after running the installation cell above and restarting the runtime.

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import os
import joblib
from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ==============================================================================
# Part 2: The Model Training Factory
# ==============================================================================

def train_all_models(season_string):
    """
    Main function to load historical data, train a suite of models for all props,
    and save the final model artifacts to Google Drive.
    """
    # --- MOUNT DRIVE AND SET UP PATHS ---
    try:
        print("Mounting Google Drive...")
        drive.mount('/content/drive', force_remount=True)

        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"

        os.makedirs(MODEL_DIR, exist_ok=True)
        print(f"Model output directory set to: {MODEL_DIR}")

    except Exception as e:
        print(f"Could not mount Google Drive. Aborting. Error: {e}")
        return

    input_filename = f"nba_training_data_{season_string}.csv"
    full_input_path = os.path.join(DATA_DIR, input_filename)

    # --- Load the historical data ---
    try:
        print(f"Loading training data from: {full_input_path}")
        df = pd.read_csv(full_input_path)
    except FileNotFoundError:
        print(f"CRITICAL ERROR: Training data not found at '{full_input_path}'. Please run Script 1 first.")
        return

    df.dropna(inplace=True)
    print(f"Loaded {len(df)} rows of training data.")

    # --- Part 1: Train REGRESSION Models ---
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']
    features = [col for col in df.columns if not col.startswith('TARGET_')]
    X = df[features]

    for prop in regression_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training REGRESSION Model for: {prop} ---")
        y = df[target_column]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        pipeline = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge())])
        param_grid = {'ridge__alpha': [100.0, 200.0, 500.0, 1000.0]}
        grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        print(f"Best Ridge alpha for {prop}: {grid_search.best_params_['ridge__alpha']}")

        predictions = best_model.predict(X_test)
        mae = mean_absolute_error(y_test, predictions)
        print(f"  - Final Model MAE for {prop}: {mae:.2f}")

        output_filename = f"nba_{prop.lower()}_ridge_model.pkl"
        full_output_path = os.path.join(MODEL_DIR, output_filename)

        if os.path.exists(full_output_path):
            confirm = input(f"  WARNING: '{output_filename}' already exists. Overwrite? (y/n): ").lower()
            if confirm != 'y':
                print(f"  Skipping save for {prop} model.")
                continue

        joblib.dump(best_model, full_output_path)
        print(f"✅ Saved {prop} model to: {full_output_path}")

    # --- Part 2: Train CLASSIFICATION Models ---
    classification_props = ['DD', 'TD']

    for prop in classification_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training CLASSIFICATION Model for: {prop} ---")
        y = df[target_column]

        if y.sum() == 0:
            print(f"  Skipping {prop} model: No positive examples found in the dataset.")
            continue

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight='balanced', max_iter=1000))])
        pipeline.fit(X_train, y_train)

        predictions_proba = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, predictions_proba)
        print(f"  - Final Model AUC for {prop}: {auc:.2%}")

        output_filename = f"nba_{prop.lower()}_logreg_model.pkl"
        full_output_path = os.path.join(MODEL_DIR, output_filename)

        if os.path.exists(full_output_path):
            confirm = input(f"  WARNING: '{output_filename}' already exists. Overwrite? (y/n): ").lower()
            if confirm != 'y':
                print(f"  Skipping save for {prop} model.")
                continue

        joblib.dump(pipeline, full_output_path)
        print(f"✅ Saved {prop} model to: {full_output_path}")

# ==============================================================================
# Part 3: Run the script
# ==============================================================================
if __name__ == "__main__":
    SEASON_TO_TRAIN_ON = "2024-25"
    train_all_models(season_string=SEASON_TO_TRAIN_ON)

Mounting Google Drive...
Mounted at /content/drive
Model output directory set to: /content/drive/MyDrive/NBA/Training Model
Loading training data from: /content/drive/MyDrive/NBA/Historical Data/nba_training_data_2024-25.csv
Loaded 23380 rows of training data.

--- Training REGRESSION Model for: PTS ---
Best Ridge alpha for PTS: 100.0
  - Final Model MAE for PTS: 5.02
✅ Saved PTS model to: /content/drive/MyDrive/NBA/Training Model/nba_pts_ridge_model.pkl

--- Training REGRESSION Model for: REB ---
Best Ridge alpha for REB: 100.0
  - Final Model MAE for REB: 2.08
✅ Saved REB model to: /content/drive/MyDrive/NBA/Training Model/nba_reb_ridge_model.pkl

--- Training REGRESSION Model for: AST ---
Best Ridge alpha for AST: 100.0
  - Final Model MAE for AST: 1.51
✅ Saved AST model to: /content/drive/MyDrive/NBA/Training Model/nba_ast_ridge_model.pkl

--- Training REGRESSION Model for: P+R ---
Best Ridge alpha for P+R: 100.0
  - Final Model MAE for P+R: 5.92
✅ Saved P+R model to: /content/driv

Script 2:  Model Training for Multiple Seasons

In [ ]:
# ==============================================================================
# SCRIPT 2, PART 2: MODEL TRAINING (CORRECTED for Multi-Season File)
# ==============================================================================
# Run this cell ONLY after running the installation cell above and restarting the runtime.

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import os
import joblib
from datetime import datetime
from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ==============================================================================
# Part 2: The Model Training Factory
# ==============================================================================

def train_all_models(seasons_trained_on):
    """
    Main function to load multi-season historical data, train a suite of models,
    and save the final model artifacts to Google Drive.
    """
    # --- MOUNT DRIVE AND SET UP PATHS ---
    try:
        print("Mounting Google Drive...")
        drive.mount('/content/drive', force_remount=True)

        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"

        os.makedirs(MODEL_DIR, exist_ok=True)
        print(f"Model output directory set to: {MODEL_DIR}")

    except Exception as e:
        print(f"Could not mount Google Drive. Aborting. Error: {e}")
        return

    # --- CHANGED: This logic now correctly builds the MULTI-SEASON filename ---
    start_year_str = seasons_trained_on[0].split('-')[0]
    end_year_str = seasons_trained_on[-1].split('-')[0] # Note: This assumes the list is sorted, which it is.
    input_filename = f"nba_training_data_multi-season_{start_year_str}-{end_year_str}.csv"
    full_input_path = os.path.join(DATA_DIR, input_filename)

    # --- Load the historical data ---
    try:
        print(f"Loading multi-season training data from: {full_input_path}")
        df = pd.read_csv(full_input_path)
    except FileNotFoundError:
        print(f"CRITICAL ERROR: Training data not found at '{full_input_path}'. Please run the corrected Script 1 first to generate it.")
        return

    df.dropna(inplace=True)
    print(f"Loaded {len(df)} rows of training data from {len(seasons_trained_on)} seasons.")

    # --- Part 1: Train REGRESSION Models ---
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']
    features = [col for col in df.columns if not col.startswith('TARGET_')]
    X = df[features]

    for prop in regression_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training REGRESSION Model for: {prop} ---")
        y = df[target_column]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        pipeline = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge())])
        param_grid = {'ridge__alpha': [100.0, 200.0, 500.0, 1000.0]}
        grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        print(f"Best Ridge alpha for {prop}: {grid_search.best_params_['ridge__alpha']}")
        predictions = best_model.predict(X_test)
        mae = mean_absolute_error(y_test, predictions)
        print(f"  - Final Model MAE for {prop}: {mae:.2f}")

        output_filename = f"nba_{prop.lower()}_ridge_model.pkl"
        full_output_path = os.path.join(MODEL_DIR, output_filename)
        joblib.dump(best_model, full_output_path)
        print(f"✅ Saved {prop} model to: {full_output_path}")

    # --- Part 2: Train CLASSIFICATION Models ---
    classification_props = ['DD', 'TD']

    for prop in classification_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training CLASSIFICATION Model for: {prop} ---")
        y = df[target_column]

        if y.nunique() < 2:
            print(f"  Skipping {prop} model: Not enough class diversity in the dataset."); continue

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight='balanced', max_iter=1000))])
        pipeline.fit(X_train, y_train)

        predictions_proba = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, predictions_proba)
        print(f"  - Final Model AUC for {prop}: {auc:.2%}")

        output_filename = f"nba_{prop.lower()}_logreg_model.pkl"
        full_output_path = os.path.join(MODEL_DIR, output_filename)
        joblib.dump(pipeline, full_output_path)
        print(f"✅ Saved {prop} model to: {full_output_path}")

# ==============================================================================
# Part 3: Run the script
# ==============================================================================
if __name__ == "__main__":
    # --- This list MUST MATCH the list you used in Script 1 ---
    SEASONS_TO_TRAIN_ON = ["2022-23", "2023-24", "2024-25"]
    print(f"--- Preparing to train models on data from seasons: {', '.join(SEASONS_TO_TRAIN_ON)} ---")
    train_all_models(seasons_trained_on=SEASONS_TO_TRAIN_ON)

--- Preparing to train models on data from seasons: 2022-23, 2023-24, 2024-25 ---
Mounting Google Drive...
Mounted at /content/drive
Model output directory set to: /content/drive/MyDrive/NBA/Training Model
Loading multi-season training data from: /content/drive/MyDrive/NBA/Historical Data/nba_training_data_multi-season_2022-2024.csv
CRITICAL ERROR: Training data not found at '/content/drive/MyDrive/NBA/Historical Data/nba_training_data_multi-season_2022-2024.csv'. Please run the corrected Script 1 first to generate it.


**SCRIPT 3:** *The Daily Prediction Engine*

Do not delete below

In [ ]:
# ==============================================================================
# SCRIPT 3: The Definitive Prediction & Multi-Sheet Output Engine
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install selenium xlsxwriter nba_api scikit-learn==1.3.0 pandas joblib numpy==1.26.4 pytz unidecode
!apt-get update --quiet
!apt-get install -y chromium-chromedriver --quiet

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import requests
import re
import time
import os
import joblib
import pytz
from datetime import datetime, timezone, timedelta
from unidecode import unidecode
from io import StringIO
from google.colab import drive
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashplayerstats, teamestimatedmetrics, playergamelog
from IPython.display import display

# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
API_TIMEOUT = 60; API_RETRY_ATTEMPTS = 3; API_RETRY_DELAY = 3; INTER_PLAYER_DELAY = 0.6
ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1",
    "146dff42ae19a83208b059d4855095c9", "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9",
    "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b", "66953d96fcb2ec72a47e7f39901258e7",
    "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1"
]

# ==============================================================================
# Part 2: Helper Functions and Data Fetching
# ==============================================================================

def normalize_player_name(name):
    """Cleans and standardizes a player's name for reliable matching."""
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name

def american_to_prob(odds):
    """Converts American odds to implied probability."""
    if odds is None or pd.isna(odds): return None
    odds = float(odds)
    if odds > 0: return 100 / (odds + 100)
    else: return abs(odds) / (abs(odds) + 100)

def scrape_bref_positions_by_team(season_end_year, teams_to_scrape, fallback_attempted=False):
    """Scrapes B-Ref team by team for a specific list of teams."""
    print(f"--- Scraping B-Ref rosters via Selenium for {len(teams_to_scrape)} teams for season ending in {season_end_year} ---")

    all_players_df_list = []
    driver = None  # Initialize driver outside the loop

    try:
        # --- IMPROVEMENT 1: Create ONE browser instance and reuse it for all teams. This is much more stable. ---
        chrome_options = Options()
        chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        driver = webdriver.Chrome(options=chrome_options)

        for i, team_abbr in enumerate(teams_to_scrape):
            try:
                print(f"Scraping {team_abbr} ({i+1}/{len(teams_to_scrape)})...")
                url = f"https://www.basketball-reference.com/teams/{team_abbr}/{season_end_year}.html"
                driver.get(url)

                # --- IMPROVEMENT 2: Use an "intelligent wait" instead of a blind sleep. ---
                # This waits up to 30 seconds for the roster table to appear before continuing.
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.ID, "roster"))
                )

                page_source = driver.page_source
                tables = pd.read_html(StringIO(page_source), attrs={'id': 'roster'})

                if tables:
                    team_df = tables[0]
                    team_df['CURRENT_TEAM'] = team_abbr
                    all_players_df_list.append(team_df)
                else:
                    raise ValueError("Roster table not found after waiting.")

            except ValueError:
                # This handles the specific "table not found" error, including the fallback.
                if i == 0 and not fallback_attempted:
                    print(f"Warning: Roster table not found for {team_abbr}. Falling back to previous season...")
                    if driver: driver.quit() # Close current driver before restarting
                    return scrape_bref_positions_by_team(season_end_year - 1, teams_to_scrape, fallback_attempted=True)
            except Exception as e:
                # This catches other errors for a single team (like timeouts) and moves on.
                print(f"  - Could not scrape {team_abbr}. Error: {e}")

    finally:
        # --- IMPROVEMENT 3: Ensure the browser is always closed cleanly at the end. ---
        if driver:
            driver.quit()

    if not all_players_df_list:
        print("CRITICAL ERROR: Failed to scrape any team data."); return pd.DataFrame()

    all_players_df = pd.concat(all_players_df_list, ignore_index=True)
    all_players_df.rename(columns={'Player': 'BRef_Name', 'Pos': 'DetailedPos'}, inplace=True)
    all_players_df = all_players_df.drop_duplicates(subset='BRef_Name', keep='first')
    print(f"\nSuccessfully scraped {len(all_players_df)} unique players from {len(all_players_df_list)} teams for season {season_end_year}.")
    return all_players_df[['BRef_Name', 'DetailedPos', 'CURRENT_TEAM']]

def fetch_todays_odds_data():
    """Follows the correct 2-step process with robust key rotation and correct timezone logic."""
    print("--- Fetching all of today's game and odds data from Odds API (Correct Method) ---")
    MARKETS_TO_KEEP = {'player_points': 'PTS', 'player_rebounds': 'REB', 'player_assists': 'AST', 'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A', 'player_double_double': 'DD', 'player_triple_double': 'TD'}

    eastern = pytz.timezone('US/Eastern'); now_et = datetime.now(eastern)
    start_of_day_et = now_et.replace(hour=8, minute=0, second=0, microsecond=0)
    end_of_day_et = start_of_day_et + timedelta(days=1)
    start_utc_str = start_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    end_utc_str = end_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    print(f"Querying for games between {start_utc_str} and {end_utc_str} (UTC)")

    working_key = None; events = []
    for i, api_key in enumerate(ODDS_API_KEYS):
        print(f"\nStep 1: Attempting to fetch event list with API Key #{i+1}...")
        try:
            response = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/events', params={'apiKey': api_key, 'commenceTimeFrom': start_utc_str, 'commenceTimeTo': end_utc_str}, timeout=30)
            remaining_reqs = response.headers.get('x-requests-remaining', 'N/A')
            print(f"  - API call successful. Requests remaining: {remaining_reqs}")
            response.raise_for_status()
            events = response.json(); working_key = api_key
            print(f"--- Successfully connected! Found {len(events)} games for today. ---"); break
        except Exception as e:
            print(f"--- API Key #{i+1} failed. Error: {e} ---")
    if not working_key:
        print("--- CRITICAL: All API keys failed. Could not fetch event list. ---"); return [], pd.DataFrame()
    if not events:
        print("No NBA games scheduled for today according to the Odds API."); return [], pd.DataFrame()

    print(f"\nStep 2: Now fetching props for each of the {len(events)} games...")
    all_props = []; games_today_set = set()
    team_name_map = {team['full_name']: team['abbreviation'] for team in teams.get_teams()}
    current_key_index = ODDS_API_KEYS.index(working_key)
    for event in events:
        event_id = event['id']
        print(f"  - Fetching props for game: {event['away_team']} @ {event['home_team']}...")
        odds_fetched = False
        while not odds_fetched and current_key_index < len(ODDS_API_KEYS):
            api_key = ODDS_API_KEYS[current_key_index]
            url = f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds'
            params = {'apiKey': api_key, 'regions': 'us', 'markets': ','.join(MARKETS_TO_KEEP.keys()), 'bookmakers': 'draftkings,fanduel'}
            try:
                response = requests.get(url, params=params, timeout=30)
                remaining_reqs = response.headers.get('x-requests-remaining', 'N/A')
                print(f"    - API call with Key #{current_key_index + 1} successful. Requests remaining: {remaining_reqs}")
                response.raise_for_status()
                event_odds = response.json()
                home_abbr = team_name_map.get(event_odds['home_team'], event_odds['home_team']); away_abbr = team_name_map.get(event_odds['away_team'], event_odds['away_team'])
                games_today_set.add((home_abbr, away_abbr))
                for bookmaker in event_odds.get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        prop_type = MARKETS_TO_KEEP.get(market['key'])
                        if not prop_type: continue
                        for outcome in market.get('outcomes', []):
                            all_props.append({'Game': f"{away_abbr} @ {home_abbr}", 'Player': outcome['description'], 'Prop': prop_type, 'Line': outcome.get('point'), 'Over_Odds': outcome['price'] if outcome['name'] == 'Over' else None, 'Under_Odds': outcome['price'] if outcome['name'] == 'Under' else None, 'Yes_Odds': outcome['price'] if outcome['name'] == 'Yes' else None, 'No_Odds': outcome['price'] if outcome['name'] == 'No' else None, 'Bookmaker': bookmaker['title']})
                odds_fetched = True
            except Exception as e:
                print(f"    - Key #{current_key_index + 1} failed for event {event_id}. Trying next key. Error: {e}")
                current_key_index += 1
        if not odds_fetched: print(f"    - WARNING: All keys failed for event {event_id}. Could not fetch props.")
        time.sleep(1)
    if not all_props:
        print("No player props were available across any markets."); return [], pd.DataFrame()
    games_today = [{'home_team_abbr': h, 'away_team_abbr': a} for h, a in games_today_set]
    df = pd.DataFrame(all_props); df['match_key'] = df['Player'].apply(normalize_player_name)
    ou_props = df[df['Prop'].isin(['PTS', 'REB', 'AST', 'PRA', 'P+R', 'P+A', 'R+A'])]
    yn_props = df[df['Prop'].isin(['DD', 'TD'])]
    odds_df_ou = ou_props.groupby(['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game']).agg({'Over_Odds': 'first', 'Under_Odds': 'first'}).reset_index()
    odds_df_yn = yn_props.groupby(['match_key', 'Player', 'Prop', 'Bookmaker', 'Game']).agg({'Yes_Odds': 'first', 'No_Odds': 'first'}).reset_index()
    odds_df_yn['Implied_Prob_Yes'] = odds_df_yn['Yes_Odds'].apply(american_to_prob)
    odds_df = pd.concat([odds_df_ou, odds_df_yn], ignore_index=True)
    print(f"\nSuccessfully parsed {len(odds_df)} unique prop lines across all available markets.")
    return games_today, odds_df

def fetch_nba_api_player_stats(season):
    """Fetches player stats from the official NBA API for a given season."""
    print(f"Fetching player stats from nba_api for season: {season}...")
    try: return leaguedashplayerstats.LeagueDashPlayerStats(season=season, per_mode_detailed="PerGame").get_data_frames()[0]
    except Exception as e: print(f"Error fetching nba_api player stats: {e}"); return pd.DataFrame()

def fetch_team_metrics(season):
    """Fetches advanced team metrics for a given season."""
    print(f"Fetching team metrics for season: {season}...")
    try:
        metrics = teamestimatedmetrics.TeamEstimatedMetrics(season=season).get_data_frames()[0]
        team_map = {team['id']: team['abbreviation'] for team in teams.get_teams()}; metrics['TEAM_ABBREVIATION'] = metrics['TEAM_ID'].map(team_map)
        return metrics
    except Exception as e: print(f"Error fetching team metrics: {e}"); return pd.DataFrame()

feature_cache = {}
def get_live_features_for_player(player_id, stats_season):
    """Fetches a player's recent game logs with delays, longer timeouts, and retries."""
    if player_id in feature_cache: return feature_cache[player_id]
    print(f"  - Engineering features for player ID: {player_id}...")
    logs = pd.DataFrame()
    for attempt in range(API_RETRY_ATTEMPTS):
        try:
            logs = playergamelog.PlayerGameLog(player_id=player_id, season=stats_season, timeout=API_TIMEOUT).get_data_frames()[0]
            break
        except Exception as e:
            print(f"    - Attempt {attempt + 1} failed for player {player_id}. Error: {e}. Retrying...")
            time.sleep(API_RETRY_DELAY)
    if logs.empty or len(logs) < 10:
        if logs.empty: print(f"    - Could not get game logs for player {player_id} after multiple attempts.")
        else: print(f"    - Warning: Player {player_id} has fewer than 10 games. Projections may be unreliable.")
        return None
    logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE']); logs = logs.sort_values('GAME_DATE', ascending=True)
    features = {}; stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']; windows = [3, 5, 10]
    for stat in stats_to_roll:
        for w in windows: features[f'ROLL_AVG_{stat}_{w}'] = logs[stat].tail(w).mean()
    last_game_date = logs.iloc[-1]['GAME_DATE']; features['DAYS_OF_REST'] = (datetime.now() - last_game_date).days
    feature_cache[player_id] = features
    return features

# ==============================================================================
# Part 4: The Main Prediction Engine
# ==============================================================================
def run_daily_projections():
    """Main function that orchestrates the entire process."""
    try:
        drive.mount('/content/drive', force_remount=True)
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"; OUTPUT_DIR = "/content/drive/MyDrive/NBA"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    print("\n--- Loading Trained Models ---")
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']; classification_props = ['DD', 'TD']
    models = {}
    try:
        for prop in regression_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_ridge_model.pkl"))
        for prop in classification_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_logreg_model.pkl"))
        print(f"Successfully loaded {len(models)} trained models.")
    except FileNotFoundError as e: print(f"CRITICAL ERROR: Could not load a model. Please run Script 2 first. Missing file: {e.filename}"); return

    games_today, odds_df = fetch_todays_odds_data()
    STATS_SEASON_STRING = "2024-25"; POSITIONS_SEASON_END_YEAR = 2026

    if not games_today:
        print("\nNo games today to analyze. The script will now exit."); return

    teams_playing_today = {g['home_team_abbr'] for g in games_today} | {g['away_team_abbr'] for g in games_today}

    bref_df = scrape_bref_positions_by_team(season_end_year=POSITIONS_SEASON_END_YEAR, teams_to_scrape=list(teams_playing_today))
    nba_df = fetch_nba_api_player_stats(season=STATS_SEASON_STRING)
    team_metrics_df = fetch_team_metrics(season=STATS_SEASON_STRING)

    if nba_df.empty or team_metrics_df.empty:
        print("\nAborting analysis due to missing critical data (NBA.com or Team Metrics)."); return

    print("\n--- Merging NBA.com stats with scraped B-Ref rosters ---")
    master_df = pd.merge(nba_df, bref_df, left_on='PLAYER_NAME', right_on='BRef_Name', how='inner')
    all_projections = []

    for game in games_today:
        home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
        teams_in_game = [home_abbr, away_abbr]
        home_metrics = team_metrics_df[team_metrics_df['TEAM_ABBREVIATION'] == home_abbr].iloc[0]
        away_metrics = team_metrics_df[team_metrics_df['TEAM_ABBREVIATION'] == away_abbr].iloc[0]
        players_to_project = master_df[master_df['CURRENT_TEAM'].isin(teams_in_game)]

        for _, player in players_to_project.iterrows():
            time.sleep(INTER_PLAYER_DELAY)
            player_features = get_live_features_for_player(player['PLAYER_ID'], STATS_SEASON_STRING)
            if not player_features: continue

            opp_metrics = away_metrics if player['CURRENT_TEAM'] == home_abbr else home_metrics
            player_features['E_PACE_OPP'] = opp_metrics['E_PACE']
            player_features['E_DEF_RATING_OPP'] = opp_metrics['E_DEF_RATING']

            projections_for_player = {'Player': player['PLAYER_NAME'], 'Team': player['CURRENT_TEAM'], 'Pos': player['DetailedPos'], 'match_key': normalize_player_name(player['PLAYER_NAME'])}

            feature_names = models['PTS'].named_steps['scaler'].get_feature_names_out()
            feature_df = pd.DataFrame([player_features])[feature_names]

            for prop in regression_props: projections_for_player[f'Proj_{prop}'] = round(models[prop].predict(feature_df)[0], 2)
            for prop in classification_props: projections_for_player[f'Proj_{prop}_%'] = f"{models[prop].predict_proba(feature_df)[0][1]:.1%}"
            all_projections.append(projections_for_player)

    projections_df = pd.DataFrame(all_projections)

    print("\n--- Generating Output Files ---")
    melted_projections = projections_df.melt(id_vars=['match_key', 'Player', 'Team', 'Pos'], value_vars=[col for col in projections_df.columns if col.startswith('Proj')], var_name='Prop_Name', value_name='Projection')
    melted_projections['Prop_Type'] = melted_projections['Prop_Name'].str.replace('Proj_', '').str.replace('_%', '')

    if odds_df.empty:
        print("WARNING: No odds data was fetched. The main report will contain projections only.")
        master_long_df = melted_projections.rename(columns={'Prop_Type': 'Prop'})
    else:
        master_long_df = pd.merge(melted_projections, odds_df, left_on=['match_key', 'Prop_Type'], right_on=['match_key', 'Prop'], how='left', suffixes=('_proj', '_odds'))
        # Consolidate Player columns
        master_long_df['Player'] = master_long_df['Player_proj'].fillna(master_long_df['Player_odds'])
        master_long_df.drop(columns=['Player_proj', 'Player_odds'], inplace=True, errors='ignore')

    ou_mask = master_long_df['Prop'].isin(regression_props); yn_mask = master_long_df['Prop'].isin(classification_props)
    master_long_df['Projection_Num'] = pd.to_numeric(master_long_df['Projection'].astype(str).str.replace('%', ''), errors='coerce')
    master_long_df.loc[ou_mask, 'Edge'] = master_long_df['Projection_Num'] - master_long_df['Line']
    master_long_df.loc[ou_mask, 'Bet'] = master_long_df['Edge'].apply(lambda x: 'Over' if x > 1.5 else ('Under' if x < -1.5 else 'Pass'))
    master_long_df.loc[yn_mask, 'Edge'] = (master_long_df['Projection_Num'] / 100) - master_long_df['Implied_Prob_Yes']
    master_long_df.loc[yn_mask, 'Bet'] = master_long_df['Edge'].apply(lambda x: 'Yes' if x > 0.10 else ('No' if x < -0.10 else 'Pass'))

    output_filename = f"{datetime.now().strftime('%m-%d-%Y')} NBA Props Predictions.xlsx"
    full_output_path = os.path.join(OUTPUT_DIR, output_filename)

    with pd.ExcelWriter(full_output_path, engine='xlsxwriter') as writer:
        print("\nWriting to Excel File...")

        if not odds_df.empty:
            ev_bets_df = master_long_df[master_long_df['Bet'].isin(['Over', 'Under', 'Yes', 'No'])].copy()
            for bookmaker in ['DraftKings', 'FanDuel']:
                book_df = ev_bets_df[ev_bets_df['Bookmaker'] == bookmaker]
                if book_df.empty: print(f"  - No EV bets found for {bookmaker}. Skipping sheet."); continue

                # THE DEFINITIVE FIX for KeyError and spacer rows
                final_ev_df_list = []
                for game_tuple in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
                    game_str = f"{game_tuple['away_team_abbr']} @ {game_tuple['home_team_abbr']}"
                    game_bets = book_df[book_df['Game'] == game_str]
                    if not game_bets.empty:
                        final_ev_df_list.append(game_bets.sort_values(by='Edge', ascending=False))
                        # Add spacer row with all columns to prevent concat issues
                        spacer_columns = game_bets.columns
                        spacer_row = pd.DataFrame([['-'*15] * len(spacer_columns)], columns=spacer_columns)
                        final_ev_df_list.append(spacer_row)

                if final_ev_df_list:
                    final_ev_df_list.pop() # Remove last spacer
                    final_ev_df = pd.concat(final_ev_df_list, ignore_index=True)
                    final_ev_df.rename(columns={'Projection_Num': 'Projection'}, inplace=True)

                    ev_cols = ['Game', 'Player', 'Team', 'Prop', 'Line', 'Projection', 'Edge', 'Bet', 'Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds']
                    for col in ev_cols:
                        if col not in final_ev_df.columns: final_ev_df[col] = None

                    final_ev_df[ev_cols].to_excel(writer, sheet_name=f"{bookmaker}_EV_Bets", index=False)
                    print(f"  - Generated sheet: {bookmaker}_EV_Bets")
        else:
            print("  - Skipping EV Bet sheets because no odds data was found.")

        for game in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
            home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
            sheet_name = f"{away_abbr}_vs_{home_abbr}"
            game_df = master_long_df[master_long_df['Team'].isin([home_abbr, away_abbr])].copy()
            if game_df.empty: continue

            # THE DEFINITIVE FIX for KeyError: 'Player' in groupby
            def summarize_player_bets(group):
                recommended = group[group['Bet'].isin(['Over', 'Under', 'Yes', 'No'])].copy()
                recommended['abs_edge'] = recommended['Edge'].abs()
                recommended = recommended.sort_values(by='abs_edge', ascending=False)
                bet_summary = ", ".join([f"{row['Prop']}: {row['Bet']} {row['Line']} (Edge: {row['Edge']:+.2f})" for _, row in recommended.iterrows()]) if not recommended.empty else "No significant edge found"
                # This ensures we get the static info without index issues
                first_row = group.iloc[0]
                return pd.Series({
                    'Edge_Count': len(recommended), 'Total_Edge_Score': recommended['abs_edge'].sum(),
                    'Recommended_Bets': bet_summary
                })

            summary_df = game_df.groupby('Player').apply(summarize_player_bets).reset_index()

            # Merge player info back to summary
            player_info = game_df[['Player', 'Team', 'Pos']].drop_duplicates(subset='Player')
            summary_df = pd.merge(player_info, summary_df, on='Player')

            pivot_proj = game_df.pivot_table(index='Player', columns='Prop_Name', values='Projection', aggfunc='first').reset_index()
            pivot_proj.columns.name = None
            renames = {f'Proj_{p}_%': f'Proj_{p}_%' for p in classification_props}
            renames.update({f'Proj_{p}': f'Proj_{p}' for p in regression_props})
            pivot_proj = pivot_proj.rename(columns=renames)

            final_game_df = pd.merge(summary_df, pivot_proj, on='Player', how='left')

            first_cols = ['Player', 'Team', 'Pos', 'Edge_Count', 'Total_Edge_Score', 'Recommended_Bets']
            proj_cols = [f'Proj_{p}' for p in ['PTS', 'REB', 'AST', 'PRA']] + [f'Proj_{p}_%' for p in ['DD', 'TD']]
            for col in first_cols + proj_cols:
                if col not in final_game_df.columns: final_game_df[col] = None

            final_game_df = final_game_df[first_cols + proj_cols].sort_values(by='Total_Edge_Score', ascending=False)

            final_game_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  - Generated sheet: {sheet_name}")

    print(f"\n✅ Successfully exported all data to: {full_output_path}")

# ==============================================================================
# Part 5: Run the script
# ==============================================================================
if __name__ == "__main__":
    pd.set_option('display.max_rows', 200); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000)
    run_daily_projections()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.6 MB/s eta 0:00:00
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease

/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_df = game_df.groupby('Player').apply(summarize_player_bets).reset_index()
/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_df = game_df.groupby('Player').apply(summarize_player_bets).reset_index()
/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.app

  - Generated sheet: DET_vs_HOU
  - Generated sheet: PHX_vs_LAC
  - Generated sheet: MIN_vs_LAL


/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_df = game_df.groupby('Player').apply(summarize_player_bets).reset_index()
/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_df = game_df.groupby('Player').apply(summarize_player_bets).reset_index()
/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.app

  - Generated sheet: MIA_vs_MEM
  - Generated sheet: SAS_vs_NOP
  - Generated sheet: BOS_vs_NYK


/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_df = game_df.groupby('Player').apply(summarize_player_bets).reset_index()
/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_df = game_df.groupby('Player').apply(summarize_player_bets).reset_index()
/tmp/ipython-input-4082821921.py:354: DeprecationWarning: DataFrameGroupBy.app

  - Generated sheet: ATL_vs_ORL
  - Generated sheet: GSW_vs_POR
  - Generated sheet: UTA_vs_SAC
  - Generated sheet: MIL_vs_TOR

✅ Successfully exported all data to: /content/drive/MyDrive/NBA/10-25-2025 NBA Props Predictions.xlsx


**Script 3:** The Definitive Prediction & Multi-Sheet Output Engine

In [ ]:
# ==============================================================================
# SCRIPT 3: The Definitive Prediction & Multi-Sheet Output Engine
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install selenium xlsxwriter nba_api scikit-learn==1.3.0 pandas joblib numpy==1.26.4 pytz unidecode requests
!apt-get update --quiet
!apt-get install -y chromium-chromedriver --quiet

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import requests
import re
import time
import os
import joblib
import pytz
from datetime import datetime, timezone, timedelta
from unidecode import unidecode
from io import StringIO
from google.colab import drive
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashplayerstats, teamestimatedmetrics, playergamelog
from IPython.display import display

# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
API_TIMEOUT = 60; API_RETRY_ATTEMPTS = 3; API_RETRY_DELAY = 3; INTER_PLAYER_DELAY = 0.6
ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1",
    "146dff42ae19a83208b059d4855095c9", "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9",
    "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b", "66953d96fcb2ec72a47e7f39901258e7",
    "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "7b64d450f91079f60211cb33033594b4"
]

# ==============================================================================
# Part 2: Helper Functions and Data Fetching
# ==============================================================================

def normalize_player_name(name):
    """Cleans and standardizes a player's name for reliable matching."""
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name

def american_to_prob(odds):
    """Converts American odds to implied probability."""
    if odds is None or pd.isna(odds): return None
    odds = float(odds)
    if odds > 0: return 100 / (odds + 100)
    else: return abs(odds) / (abs(odds) + 100)

def scrape_bref_positions_by_team(season_end_year, teams_to_scrape, fallback_attempted=False):
    """Scrapes B-Ref team by team for a specific list of teams."""
    print(f"--- Scraping B-Ref rosters via Selenium for {len(teams_to_scrape)} teams for season ending in {season_end_year} ---")
    all_players_df_list = []
    for i, team_abbr in enumerate(teams_to_scrape):
        driver = None
        try:
            print(f"Scraping {team_abbr} ({i+1}/{len(teams_to_scrape)})...")
            chrome_options = Options(); chrome_options.add_argument('--headless'); chrome_options.add_argument('--no-sandbox'); chrome_options.add_argument('--disable-dev-shm-usage')
            driver = webdriver.Chrome(options=chrome_options)
            driver.set_page_load_timeout(API_TIMEOUT)
            url = f"https://www.basketball-reference.com/teams/{team_abbr}/{season_end_year}.html"
            driver.get(url)
            time.sleep(2)
            page_source = driver.page_source
            tables = pd.read_html(StringIO(page_source), attrs={'id': 'roster'})
            if tables:
                team_df = tables[0]; team_df['CURRENT_TEAM'] = team_abbr
                all_players_df_list.append(team_df)
            else: raise ValueError("Roster table not found.")
        except ValueError:
            if i == 0 and not fallback_attempted:
                print(f"Warning: Roster table not found for {team_abbr}. Falling back to previous season...")
                if driver: driver.quit()
                return scrape_bref_positions_by_team(season_end_year - 1, teams_to_scrape, fallback_attempted=True)
        except Exception as e:
            print(f"  - Could not scrape {team_abbr}. Error: {e}")
        finally:
            if driver: driver.quit()
    if not all_players_df_list:
        print("CRITICAL ERROR: Failed to scrape any team data."); return pd.DataFrame()
    all_players_df = pd.concat(all_players_df_list, ignore_index=True)
    all_players_df.rename(columns={'Player': 'BRef_Name', 'Pos': 'DetailedPos'}, inplace=True)
    all_players_df = all_players_df.drop_duplicates(subset='BRef_Name', keep='first')
    print(f"\nSuccessfully scraped {len(all_players_df)} unique players from {len(all_players_df_list)} teams for season {season_end_year}.")
    return all_players_df[['BRef_Name', 'DetailedPos', 'CURRENT_TEAM']]

def fetch_todays_odds_data():
    """Follows the correct 2-step process with robust key rotation and correct timezone logic."""
    print("--- Fetching all of today's game and odds data from Odds API (Correct Method) ---")
    MARKETS_TO_KEEP = {'player_points': 'PTS', 'player_rebounds': 'REB', 'player_assists': 'AST', 'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A', 'player_double_double': 'DD', 'player_triple_double': 'TD'}

    eastern = pytz.timezone('US/Eastern'); now_et = datetime.now(eastern)
    start_of_day_et = now_et.replace(hour=8, minute=0, second=0, microsecond=0)
    end_of_day_et = start_of_day_et + timedelta(days=1)
    start_utc_str = start_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    end_utc_str = end_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    print(f"Querying for games between {start_utc_str} and {end_utc_str} (UTC)")

    working_key = None; events = []
    for i, api_key in enumerate(ODDS_API_KEYS):
        print(f"\nStep 1: Attempting to fetch event list with API Key #{i+1}...")
        try:
            response = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/events', params={'apiKey': api_key, 'commenceTimeFrom': start_utc_str, 'commenceTimeTo': end_utc_str}, timeout=30)
            remaining_reqs = response.headers.get('x-requests-remaining', 'N/A')
            print(f"  - API call successful. Requests remaining: {remaining_reqs}")
            response.raise_for_status()
            events = response.json(); working_key = api_key
            print(f"--- Successfully connected! Found {len(events)} games for today. ---"); break
        except Exception as e:
            print(f"--- API Key #{i+1} failed. Error: {e} ---")
    if not working_key:
        print("--- CRITICAL: All API keys failed. Could not fetch event list. ---"); return [], pd.DataFrame()
    if not events:
        print("No NBA games scheduled for today according to the Odds API."); return [], pd.DataFrame()

    print(f"\nStep 2: Now fetching props for each of the {len(events)} games...")
    all_props = []; games_today_set = set()
    team_name_map = {team['full_name']: team['abbreviation'] for team in teams.get_teams()}
    current_key_index = ODDS_API_KEYS.index(working_key)
    for event in events:
        event_id = event['id']
        commence_time = event['commence_time']
        print(f"  - Fetching props for game: {event['away_team']} @ {event['home_team']}...")
        odds_fetched = False
        while not odds_fetched and current_key_index < len(ODDS_API_KEYS):
            api_key = ODDS_API_KEYS[current_key_index]
            url = f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds'
            params = {'apiKey': api_key, 'regions': 'us', 'markets': ','.join(MARKETS_TO_KEEP.keys()), 'bookmakers': 'draftkings,fanduel'}
            try:
                response = requests.get(url, params=params, timeout=30)
                remaining_reqs = response.headers.get('x-requests-remaining', 'N/A')
                print(f"    - API call with Key #{current_key_index + 1} successful. Requests remaining: {remaining_reqs}")
                response.raise_for_status()
                event_odds = response.json()
                home_abbr = team_name_map.get(event_odds['home_team'], event_odds['home_team']); away_abbr = team_name_map.get(event_odds['away_team'], event_odds['away_team'])
                games_today_set.add((home_abbr, away_abbr))
                for bookmaker in event_odds.get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        prop_type = MARKETS_TO_KEEP.get(market['key'])
                        if not prop_type: continue
                        for outcome in market.get('outcomes', []):
                            all_props.append({'Game': f"{away_abbr} @ {home_abbr}", 'Game_Time': commence_time, 'Player': outcome['description'], 'Prop': prop_type, 'Line': outcome.get('point'), 'Over_Odds': outcome['price'] if outcome['name'] == 'Over' else None, 'Under_Odds': outcome['price'] if outcome['name'] == 'Under' else None, 'Yes_Odds': outcome['price'] if outcome['name'] == 'Yes' else None, 'No_Odds': outcome['price'] if outcome['name'] == 'No' else None, 'Bookmaker': bookmaker['title']})
                odds_fetched = False
            except Exception as e:
                print(f"    - Key #{current_key_index + 1} failed for event {event_id}. Trying next key. Error: {e}")
                current_key_index += 1
        if not odds_fetched: print(f"    - WARNING: All keys failed for event {event_id}. Could not fetch props.")
        time.sleep(1)
    if not all_props:
        print("No player props were available across any markets."); return [], pd.DataFrame()
    games_today = [{'home_team_abbr': h, 'away_team_abbr': a} for h, a in games_today_set]
    df = pd.DataFrame(all_props); df['match_key'] = df['Player'].apply(normalize_player_name)
    ou_props = df[df['Prop'].isin(['PTS', 'REB', 'AST', 'PRA', 'P+R', 'P+A', 'R+A'])]
    yn_props = df[df['Prop'].isin(['DD', 'TD'])]
    odds_df_ou = ou_props.groupby(['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time']).agg({'Over_Odds': 'first', 'Under_Odds': 'first'}).reset_index()
    odds_df_yn = yn_props.groupby(['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time']).agg({'Yes_Odds': 'first', 'No_Odds': 'first'}).reset_index()
    odds_df_yn['Implied_Prob_Yes'] = odds_df_yn['Yes_Odds'].apply(american_to_prob)
    odds_df = pd.concat([odds_df_ou, odds_df_yn], ignore_index=True)
    odds_df['Game_Time'] = pd.to_datetime(odds_df['Game_Time']).dt.tz_convert('US/Eastern').dt.strftime('%I:%M %p EST')
    print(f"\nSuccessfully parsed {len(odds_df)} unique prop lines across all available markets.")
    return games_today, odds_df

def fetch_nba_api_player_stats(season):
    """Fetches player stats from the official NBA API for a given season."""
    print(f"Fetching player stats from nba_api for season: {season}...")
    try: return leaguedashplayerstats.LeagueDashPlayerStats(season=season, per_mode_detailed="PerGame").get_data_frames()[0]
    except Exception as e: print(f"Error fetching nba_api player stats: {e}"); return pd.DataFrame()

def fetch_team_metrics(season):
    """Fetches advanced team metrics for a given season."""
    print(f"Fetching team metrics for season: {season}...")
    try:
        metrics = teamestimatedmetrics.TeamEstimatedMetrics(season=season).get_data_frames()[0]
        team_map = {team['id']: team['abbreviation'] for team in teams.get_teams()}; metrics['TEAM_ABBREVIATION'] = metrics['TEAM_ID'].map(team_map)
        return metrics
    except Exception as e: print(f"Error fetching team metrics: {e}"); return pd.DataFrame()

feature_cache = {}
def get_live_features_for_player(player_id, stats_season):
    """Fetches a player's recent game logs with delays, longer timeouts, and retries."""
    if player_id in feature_cache: return feature_cache[player_id]
    print(f"  - Engineering features for player ID: {player_id}...")
    logs = pd.DataFrame()
    for attempt in range(API_RETRY_ATTEMPTS):
        try:
            logs = playergamelog.PlayerGameLog(player_id=player_id, season=stats_season, timeout=API_TIMEOUT).get_data_frames()[0]
            break
        except Exception as e:
            print(f"    - Attempt {attempt + 1} failed for player {player_id}. Error: {e}. Retrying...")
            time.sleep(API_RETRY_DELAY)
    if logs.empty or len(logs) < 10:
        if logs.empty: print(f"    - Could not get game logs for player {player_id} after multiple attempts.")
        else: print(f"    - Warning: Player {player_id} has fewer than 10 games. Projections may be unreliable.")
        return None
    logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE']); logs = logs.sort_values('GAME_DATE', ascending=True)
    features = {}; stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']; windows = [3, 5, 10]
    for stat in stats_to_roll:
        for w in windows: features[f'ROLL_AVG_{stat}_{w}'] = logs[stat].tail(w).mean()
    last_game_date = logs.iloc[-1]['GAME_DATE']; features['DAYS_OF_REST'] = (datetime.now() - last_game_date).days
    feature_cache[player_id] = features
    return features

# ==============================================================================
# Part 4: The Main Prediction Engine
# ==============================================================================
def run_daily_projections():
    """Main function that orchestrates the entire process."""
    try:
        drive.mount('/content/drive', force_remount=True)
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"; OUTPUT_DIR = "/content/drive/MyDrive/NBA"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    print("\n--- Loading Trained Models ---")
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']; classification_props = ['DD', 'TD']
    models = {}
    try:
        for prop in regression_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_ridge_model.pkl"))
        for prop in classification_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_logreg_model.pkl"))
        print(f"Successfully loaded {len(models)} trained models.")
    except FileNotFoundError as e: print(f"CRITICAL ERROR: Could not load a model. Please run Script 2 first. Missing file: {e.filename}"); return

    games_today, odds_df = fetch_todays_odds_data()
    STATS_SEASON_STRING = "2024-25"; POSITIONS_SEASON_END_YEAR = 2026

    if not games_today:
        print("\nNo games today to analyze. The script will now exit."); return

    teams_playing_today = {g['home_team_abbr'] for g in games_today} | {g['away_team_abbr'] for g in games_today}

    bref_df = scrape_bref_positions_by_team(season_end_year=POSITIONS_SEASON_END_YEAR, teams_to_scrape=list(teams_playing_today))
    nba_df = fetch_nba_api_player_stats(season=STATS_SEASON_STRING)
    team_metrics_df = fetch_team_metrics(season=STATS_SEASON_STRING)

    if nba_df.empty or bref_df.empty or team_metrics_df.empty:
        print("\nAborting analysis due to missing critical data."); return

    print("\n--- Generating Player Projections ---")
    master_df = pd.merge(nba_df, bref_df, left_on='PLAYER_NAME', right_on='BRef_Name', how='inner')
    all_projections = []

    for game in games_today:
        home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
        teams_in_game = [home_abbr, away_abbr]
        home_metrics = team_metrics_df[team_metrics_df['TEAM_ABBREVIATION'] == home_abbr].iloc[0]
        away_metrics = team_metrics_df[team_metrics_df['TEAM_ABBREVIATION'] == away_abbr].iloc[0]
        players_to_project = master_df[master_df['CURRENT_TEAM'].isin(teams_in_game)]

        for _, player in players_to_project.iterrows():
            time.sleep(INTER_PLAYER_DELAY)
            player_features = get_live_features_for_player(player['PLAYER_ID'], STATS_SEASON_STRING)
            if not player_features: continue

            opp_metrics = away_metrics if player['CURRENT_TEAM'] == home_abbr else home_metrics
            player_features['E_PACE_OPP'] = opp_metrics['E_PACE']
            player_features['E_DEF_RATING_OPP'] = opp_metrics['E_DEF_RATING']

            projections_for_player = {'Player': player['PLAYER_NAME'], 'Team': player['CURRENT_TEAM'], 'Pos': player['DetailedPos'], 'match_key': normalize_player_name(player['PLAYER_NAME'])}

            feature_names = models['PTS'].named_steps['scaler'].get_feature_names_out()
            feature_df = pd.DataFrame([player_features])[feature_names]

            for prop in regression_props:
                pred = models[prop].predict(feature_df)[0]
                projections_for_player[f'Proj_{prop}'] = round(max(0, pred), 2)
            for prop in classification_props:
                projections_for_player[f'Proj_{prop}_%'] = f"{models[prop].predict_proba(feature_df)[0][1]:.1%}"
            all_projections.append(projections_for_player)

    projections_df = pd.DataFrame(all_projections)

    print("\n--- Generating Output Files ---")
    melted_projections = projections_df.melt(id_vars=['match_key', 'Player', 'Team', 'Pos'], value_vars=[col for col in projections_df.columns if col.startswith('Proj')], var_name='Prop_Name', value_name='Projection')
    melted_projections.rename(columns={'Prop_Type': 'Prop'}, inplace=True, errors='ignore') # To be safe
    melted_projections['Prop'] = melted_projections['Prop_Name'].str.replace('Proj_', '').str.replace('_%', '')

    if odds_df.empty:
        print("WARNING: No odds data was fetched. The main report will contain projections only.")
        master_long_df = melted_projections
    else:
        # Definitive fix for the merge KeyErrors
        master_long_df = pd.merge(melted_projections, odds_df, on=['match_key', 'Prop'], how='left', suffixes=('_proj', '_odds'))
        master_long_df['Player'] = master_long_df['Player_proj'].fillna(master_long_df['Player_odds'])
        master_long_df.drop(columns=['Player_proj', 'Player_odds'], inplace=True, errors='ignore')

    ou_mask = master_long_df['Prop'].isin(regression_props); yn_mask = master_long_df['Prop'].isin(classification_props)
    master_long_df['Projection_Num'] = pd.to_numeric(master_long_df['Projection'].astype(str).str.replace('%', ''), errors='coerce')
    master_long_df.drop(columns=['Projection'], inplace=True, errors='ignore')
    master_long_df.loc[ou_mask, 'Edge'] = master_long_df['Projection_Num'] - master_long_df['Line']
    master_long_df.loc[ou_mask, 'Bet'] = master_long_df['Edge'].apply(lambda x: 'Over' if x > 1.0 else ('Under' if x < -1.0 else 'Pass'))
    master_long_df.loc[yn_mask, 'Edge'] = (master_long_df['Projection_Num'] / 100) - master_long_df['Implied_Prob_Yes']
    master_long_df.loc[yn_mask, 'Bet'] = master_long_df['Edge'].apply(lambda x: 'Yes' if x > 0.05 else ('No' if x < -0.05 else 'Pass'))

    output_filename = f"{datetime.now().strftime('%m-%d-%Y')} NBA Props Predictions.xlsx"
    full_output_path = os.path.join(OUTPUT_DIR, output_filename)

    with pd.ExcelWriter(full_output_path, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_format = workbook.add_format({'bold': True, 'text_wrap': False, 'valign': 'top', 'fg_color': '#D7E4BC', 'border': 1, 'align': 'center'})
        center_format = workbook.add_format({'align': 'center'})

        print("\nWriting to Excel File...")

        if not odds_df.empty:
            ev_bets_df = master_long_df[master_long_df['Bet'].isin(['Over', 'Under', 'Yes', 'No'])].copy()
            for bookmaker in ['DraftKings', 'FanDuel']:
                book_df = ev_bets_df[ev_bets_df['Bookmaker'] == bookmaker]
                if book_df.empty: print(f"  - No EV bets found for {bookmaker}. Skipping sheet."); continue

                # THE DEFINITIVE FIX for KeyError and spacer rows
                final_ev_df_list = []
                for game_tuple in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
                    game_str = f"{game_tuple['away_team_abbr']} @ {game_tuple['home_team_abbr']}"
                    game_bets = book_df[book_df['Game'] == game_str]
                    if not game_bets.empty:
                        final_ev_df_list.append(game_bets.sort_values(by='Edge', ascending=False))
                        # Correctly create a spacer DataFrame that preserves columns
                        spacer = pd.DataFrame([pd.NA] * len(game_bets.columns), index=game_bets.columns).T
                        final_ev_df_list.append(spacer)

                if final_ev_df_list:
                    if len(final_ev_df_list) > 0: final_ev_df_list.pop() # Remove last spacer
                    final_ev_df = pd.concat(final_ev_df_list, ignore_index=True)
                    final_ev_df.rename(columns={'Projection_Num': 'Projection'}, inplace=True)
                    ev_cols = ['Game_Time', 'Game', 'Player', 'Team', 'Prop', 'Line', 'Projection', 'Edge', 'Bet', 'Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds']
                    final_ev_df = final_ev_df.reindex(columns=ev_cols) # Safely select and order columns

                    sheet_name = f"{bookmaker}_EV_Bets"
                    final_ev_df.to_excel(writer, sheet_name=sheet_name, index=False)
                    worksheet = writer.sheets[sheet_name]
                    for col_num, value in enumerate(final_ev_df.columns.values): worksheet.write(0, col_num, value, header_format)
                    worksheet.autofit(); worksheet.freeze_panes(1, 0)
                    print(f"  - Generated sheet: {sheet_name}")
        else:
            print("  - Skipping EV Bet sheets because no odds data was found.")

        # Game-by-Game "Full Data" Sheets
        for game in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
            home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
            sheet_name = f"{away_abbr}_vs_{home_abbr}_Full_Data"
            game_df = master_long_df[(master_long_df['Team'] == home_abbr) | (master_long_df['Team'] == away_abbr)].copy()
            if game_df.empty: continue

            game_df.rename(columns={'Projection_Num': 'Projection'}, inplace=True)
            final_cols = ['Player', 'Team', 'Prop', 'Projection', 'Line', 'Edge', 'Bet', 'Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds', 'Bookmaker']
            game_long_df = game_df.reindex(columns=final_cols).sort_values(by=['Player', 'Prop'])

            game_long_df.to_excel(writer, sheet_name=sheet_name, index=False)
            worksheet = writer.sheets[sheet_name]
            for col_num, value in enumerate(game_long_df.columns.values): worksheet.write(0, col_num, value, header_format)
            worksheet.autofit(); worksheet.freeze_panes(1, 0)
            print(f"  - Generated sheet: {sheet_name}")

    print(f"\n✅ Successfully exported all data to: {full_output_path}")

# ==============================================================================
# Part 5: Run the script
# ==============================================================================
if __name__ == "__main__":
    pd.set_option('display.max_rows', 200); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000)
    run_daily_projections()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 81.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 39.1 MB/s eta 0:00:00
  Created wheel for scikit-learn: filename=scikit_learn-1.3.0-cp312-cp312-linux_x86_64.whl size=11156322 sha256=b485735e6600be68b0f96157ff622967c547ec4daacd518042fa63df0ff96011
  Stored in directory: /root/.cache/pip/wheels/

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.2 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,411 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,289 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:14

RuntimeError: empty_like method already has a different docstring

**SCRIPT 1:** The Historical Data Engine (CORRECTED for Multi-Season)

In [ ]:
# ==============================================================================
# SCRIPT 1: The Historical Data Engine (CORRECTED, ROBUST & Multi-Season)
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install nba_api pandas unidecode
!apt-get update --quiet
!apt-get install -y chromium-chromedriver --quiet

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import re
import time
import os
import random # Import the random library for randomized sleep
from datetime import datetime
from unidecode import unidecode
from io import StringIO
from google.colab import drive
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelogs, teamestimatedmetrics
from IPython.display import display

# ==============================================================================
# --- CONFIGURATION FOR ROBUST REQUESTS ---
# ==============================================================================
API_TIMEOUT = 60  # Increased timeout to 60 seconds
API_RETRY_ATTEMPTS = 3
API_RETRY_DELAY_BASE = 10 # Base delay of 10 seconds before retrying

# ==============================================================================
# Part 2: Helper Functions (Now More Robust)
# ==============================================================================

def normalize_player_name(name):
    """Cleans and standardizes a player's name for reliable matching."""
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name

# --- MODIFIED: Function now includes a retry loop and increased timeout ---
def fetch_team_metrics(season):
    """Fetches advanced team metrics for a given season with a retry mechanism."""
    print(f"Fetching team metrics for season: {season}...")
    for attempt in range(API_RETRY_ATTEMPTS):
        try:
            metrics = teamestimatedmetrics.TeamEstimatedMetrics(season=season, timeout=API_TIMEOUT).get_data_frames()[0]
            team_map = {team['id']: team['abbreviation'] for team in teams.get_teams()}
            metrics['TEAM_ABBREVIATION'] = metrics['TEAM_ID'].map(team_map)
            print(f"  - Successfully fetched team metrics for {season}.")
            return metrics
        except Exception as e:
            print(f"  - Attempt {attempt + 1}/{API_RETRY_ATTEMPTS} failed. Error: {e}. Retrying after a delay...")
            time.sleep(API_RETRY_DELAY_BASE * (attempt + 1)) # Increase delay each time
    print(f"  - CRITICAL: Could not fetch team metrics for {season} after multiple attempts.")
    return pd.DataFrame()

# ==============================================================================
# Part 3: The Historical Data Generation Engine
# ==============================================================================

def build_historical_dataset(seasons_to_build):
    """
    Main function to build a complete training dataset from a list of seasons,
    combining them into a single file with all features and targets.
    """
    try:
        print("Mounting Google Drive..."); drive.mount('/content/drive', force_remount=True)
        OUTPUT_DIR = "/content/drive/MyDrive/NBA/Historical Data"; os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"Output directory set to: {OUTPUT_DIR}")
    except Exception as e:
        print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    start_year_str = seasons_to_build[0].split('-')[0]; end_year_str = seasons_to_build[-1].split('-')[0]
    output_filename = f"nba_training_data_multi-season_{start_year_str}-{end_year_str}.csv"
    full_output_path = os.path.join(OUTPUT_DIR, output_filename)

    if os.path.exists(full_output_path):
        confirm = input(f"WARNING: '{output_filename}' already exists. Overwrite? (y/n): ").lower()
        if confirm != 'y':
            print("Aborting script."); return
        print("Proceeding with overwrite...")

    all_seasons_data = []

    for season_string in seasons_to_build:
        print(f"\n--- Processing data for season: {season_string} ---")

        team_metrics_df = fetch_team_metrics(season=season_string)

        print(f"Fetching all player game logs for {season_string}...")
        all_gamelogs = pd.DataFrame()
        # --- ADDED: Retry loop for the main gamelog fetch ---
        for attempt in range(API_RETRY_ATTEMPTS):
            try:
                # PlayerGameLogs is efficient but needs the longer timeout
                all_gamelogs = playergamelogs.PlayerGameLogs(season_nullable=season_string, timeout=API_TIMEOUT).get_data_frames()[0]
                if not all_gamelogs.empty:
                    break # Success, exit the loop
            except Exception as e:
                print(f"  - Attempt {attempt + 1}/{API_RETRY_ATTEMPTS} failed for gamelogs. Error: {e}. Retrying after a delay...")
                time.sleep(API_RETRY_DELAY_BASE * (attempt + 1)) # Exponential backoff

        if all_gamelogs.empty:
            print(f"  - Could not fetch game logs for {season_string} after multiple attempts. Skipping season.")
            continue

        # --- ADDED: A randomized respectful sleep timer between major API calls ---
        sleep_time = random.uniform(5.0, 10.0)
        print(f"  - Successfully fetched gamelogs. Sleeping for {sleep_time:.2f} seconds before next season...")
        time.sleep(sleep_time)

        MINIMUM_MINUTES_THRESHOLD = 15; df = all_gamelogs[all_gamelogs['MIN'] >= MINIMUM_MINUTES_THRESHOLD].copy()
        print(f"  - Kept {len(df)} of {len(all_gamelogs)} entries after filtering for minutes.")
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']); df.sort_values(by=['PLAYER_ID', 'GAME_DATE'], inplace=True)

        print("  - Engineering features: Rolling Averages and Days of Rest...")
        stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']; windows = [3, 5, 10]
        grouped = df.groupby('PLAYER_ID')
        for stat in stats_to_roll:
            for w in windows:
                df[f'ROLL_AVG_{stat}_{w}'] = grouped[stat].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        df['DAYS_OF_REST'] = grouped['GAME_DATE'].transform(lambda x: x.diff().dt.days - 1).fillna(7)

        print("  - Engineering features: Opponent Metrics...")
        df['OPPONENT_ABBR'] = df['MATCHUP'].apply(lambda x: x[-3:] if isinstance(x, str) else None)
        team_metrics_df_opp = team_metrics_df.add_suffix('_OPP')
        df = pd.merge(df, team_metrics_df_opp, left_on='OPPONENT_ABBR', right_on='TEAM_ABBREVIATION_OPP', how='left')

        print("  - Defining all target variables...")
        df['TARGET_PTS'] = df['PTS']; df['TARGET_REB'] = df['REB']; df['TARGET_AST'] = df['AST']
        df['TARGET_P+R'] = df['PTS'] + df['REB']; df['TARGET_P+A'] = df['PTS'] + df['AST']; df['TARGET_R+A'] = df['REB'] + df['AST']
        df['TARGET_PRA'] = df['PTS'] + df['REB'] + df['AST']
        stat_columns = ['PTS', 'REB', 'AST', 'STL', 'BLK']; df['DD_COUNT'] = df[stat_columns].ge(10).sum(axis=1)
        df['TARGET_DD'] = (df['DD_COUNT'] >= 2).astype(int); df['TARGET_TD'] = (df['DD_COUNT'] >= 3).astype(int)
        all_seasons_data.append(df)

    if not all_seasons_data:
        print("\nCRITICAL ERROR: No data could be processed for any season. Aborting."); return

    final_df_combined = pd.concat(all_seasons_data, ignore_index=True); final_df_combined.dropna(inplace=True)
    feature_columns = [
        'ROLL_AVG_PTS_3', 'ROLL_AVG_REB_3', 'ROLL_AVG_AST_3', 'ROLL_AVG_PTS_5', 'ROLL_AVG_REB_5', 'ROLL_AVG_AST_5',
        'ROLL_AVG_PTS_10', 'ROLL_AVG_REB_10', 'ROLL_AVG_AST_10', 'ROLL_AVG_MIN_10', 'ROLL_AVG_FGA_10',
        'ROLL_AVG_FTA_10', 'ROLL_AVG_FG3A_10', 'DAYS_OF_REST', 'E_PACE_OPP', 'E_DEF_RATING_OPP'
    ]
    target_columns = [
        'TARGET_PTS', 'TARGET_REB', 'TARGET_AST', 'TARGET_P+R', 'TARGET_P+A', 'TARGET_R+A', 'TARGET_PRA',
        'TARGET_DD', 'TARGET_TD'
    ]
    final_df = final_df_combined[feature_columns + target_columns]
    final_df.to_csv(full_output_path, index=False)
    print(f"\n✅ Historical dataset build complete for {len(seasons_to_build)} seasons.")
    print(f"Generated {len(final_df)} total training rows."); print(f"File saved to: {full_output_path}")
    display(final_df.head())

# ==============================================================================
# Part 5: Run the script
# ==============================================================================
if __name__ == "__main__":
    SEASONS_TO_BUILD = ["2022-23", "2023-24", "2024-25"]
    build_historical_dataset(seasons_to_build=SEASONS_TO_BUILD)

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,086 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 2,088 kB in 1s (1,465 kB/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources

,ROLL_AVG_PTS_3,ROLL_AVG_REB_3,ROLL_AVG_AST_3,ROLL_AVG_PTS_5,ROLL_AVG_REB_5,ROLL_AVG_AST_5,ROLL_AVG_PTS_10,ROLL_AVG_REB_10,ROLL_AVG_AST_10,ROLL_AVG_MIN_10,ROLL_AVG_FGA_10,ROLL_AVG_FTA_10,ROLL_AVG_FG3A_10,DAYS_OF_REST,E_PACE_OPP,E_DEF_RATING_OPP,TARGET_PTS,TARGET_REB,TARGET_AST,TARGET_P+R,TARGET_P+A,TARGET_R+A,TARGET_PRA,TARGET_DD,TARGET_TD
161,4.000000,1.000000,2.000000,4.000000,1.0,2.00,4.000000,1.0,2.00,15.683333,7.00,5.000000,2.0,1.0,100.8,110.1,23,3,4,26,27,7,30,0,0
162,13.500000,2.000000,3.000000,13.500000,2.0,3.00,13.500000,2.0,3.00,16.558333,9.00,5.000000,4.0,6.0,103.2,111.3,25,11,3,36,28,14,39,1,0
163,17.333333,5.000000,3.000000,17.333333,5.0,3.00,17.333333,5.0,3.00,19.272222,10.00,5.333333,4.0,1.0,102.4,114.4,10,5,0,15,10,5,15,0,0
164,19.333333,6.333333,2.333333,15.500000,5.0,2.25,15.500000,5.0,2.25,18.900000,10.75,4.750000,4.0,3.0,104.0,111.8,31,15,8,46,39,23,54,1,0
165,22.000000,10.333333,3.666667,18.600000,7.0,3.40,18.600000,7.0,3.40,22.160000,13.80,4.600000,5.2,1.0,100.3,111.9,20,10,6,30,26,16,36,1,0


**SCRIPT 2**, PART 2: MODEL TRAINING (CORRECTED)

In [ ]:
# ==============================================================================
# SCRIPT 2, PART 1: INSTALLATION
# ==============================================================================
# This cell's only job is to set up the correct library versions.

!pip install scikit-learn==1.3.0 pandas joblib numpy==1.26.4

print("\n" + "="*60)
print("--> IMPORTANT: INSTALLATION COMPLETE. <--" )
print("--> You MUST restart the runtime now before running the next cell. <--" )
print("--> Go to the 'Runtime' menu above and select 'Restart session'. <--" )
print("="*60)

# This line will automatically crash the kernel to force a restart.
# After it crashes, you are ready to run the second cell.
import os
os.kill(os.getpid(), 9)

In [ ]:
# ==============================================================================
# SCRIPT 2, PART 2: MODEL TRAINING (CORRECTED for Multi-Season File)
# ==============================================================================
# Run this cell ONLY after running the installation cell above and restarting the runtime.

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import os
import joblib
from datetime import datetime
from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ==============================================================================
# Part 2: The Model Training Factory
# ==============================================================================

def train_all_models(seasons_trained_on):
    """
    Main function to load multi-season historical data, train a suite of models,
    and save the final model artifacts to Google Drive.
    """
    # --- MOUNT DRIVE AND SET UP PATHS ---
    try:
        print("Mounting Google Drive...")
        drive.mount('/content/drive', force_remount=True)

        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"

        os.makedirs(MODEL_DIR, exist_ok=True)
        print(f"Model output directory set to: {MODEL_DIR}")

    except Exception as e:
        print(f"Could not mount Google Drive. Aborting. Error: {e}")
        return

    # --- CHANGED: This logic now correctly builds the MULTI-SEASON filename ---
    start_year_str = seasons_trained_on[0].split('-')[0]
    end_year_str = seasons_trained_on[-1].split('-')[0] # Note: This assumes the list is sorted, which it is.
    input_filename = f"nba_training_data_multi-season_{start_year_str}-{end_year_str}.csv"
    full_input_path = os.path.join(DATA_DIR, input_filename)

    # --- Load the historical data ---
    try:
        print(f"Loading multi-season training data from: {full_input_path}")
        df = pd.read_csv(full_input_path)
    except FileNotFoundError:
        print(f"CRITICAL ERROR: Training data not found at '{full_input_path}'. Please run the corrected Script 1 first to generate it.")
        return

    df.dropna(inplace=True)
    print(f"Loaded {len(df)} rows of training data from {len(seasons_trained_on)} seasons.")

    # --- Part 1: Train REGRESSION Models ---
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']
    features = [col for col in df.columns if not col.startswith('TARGET_')]
    X = df[features]

    for prop in regression_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training REGRESSION Model for: {prop} ---")
        y = df[target_column]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        pipeline = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge())])
        param_grid = {'ridge__alpha': [100.0, 200.0, 500.0, 1000.0]}
        grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        print(f"Best Ridge alpha for {prop}: {grid_search.best_params_['ridge__alpha']}")
        predictions = best_model.predict(X_test)
        mae = mean_absolute_error(y_test, predictions)
        print(f"  - Final Model MAE for {prop}: {mae:.2f}")

        output_filename = f"nba_{prop.lower()}_ridge_model.pkl"
        full_output_path = os.path.join(MODEL_DIR, output_filename)
        joblib.dump(best_model, full_output_path)
        print(f"✅ Saved {prop} model to: {full_output_path}")

    # --- Part 2: Train CLASSIFICATION Models ---
    classification_props = ['DD', 'TD']

    for prop in classification_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training CLASSIFICATION Model for: {prop} ---")
        y = df[target_column]

        if y.nunique() < 2:
            print(f"  Skipping {prop} model: Not enough class diversity in the dataset."); continue

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight='balanced', max_iter=1000))])
        pipeline.fit(X_train, y_train)

        predictions_proba = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, predictions_proba)
        print(f"  - Final Model AUC for {prop}: {auc:.2%}")

        output_filename = f"nba_{prop.lower()}_logreg_model.pkl"
        full_output_path = os.path.join(MODEL_DIR, output_filename)
        joblib.dump(pipeline, full_output_path)
        print(f"✅ Saved {prop} model to: {full_output_path}")

# ==============================================================================
# Part 3: Run the script
# ==============================================================================
if __name__ == "__main__":
    # --- This list MUST MATCH the list you used in Script 1 ---
    SEASONS_TO_TRAIN_ON = ["2022-23", "2023-24", "2024-25"]
    print(f"--- Preparing to train models on data from seasons: {', '.join(SEASONS_TO_TRAIN_ON)} ---")
    train_all_models(seasons_trained_on=SEASONS_TO_TRAIN_ON)

--- Preparing to train models on data from seasons: 2022-23, 2023-24, 2024-25 ---
Mounting Google Drive...
Mounted at /content/drive
Model output directory set to: /content/drive/MyDrive/NBA/Training Model
Loading multi-season training data from: /content/drive/MyDrive/NBA/Historical Data/nba_training_data_multi-season_2022-2024.csv
Loaded 69559 rows of training data from 3 seasons.

--- Training REGRESSION Model for: PTS ---
Best Ridge alpha for PTS: 100.0
  - Final Model MAE for PTS: 5.04
✅ Saved PTS model to: /content/drive/MyDrive/NBA/Training Model/nba_pts_ridge_model.pkl

--- Training REGRESSION Model for: REB ---
Best Ridge alpha for REB: 100.0
  - Final Model MAE for REB: 2.04
✅ Saved REB model to: /content/drive/MyDrive/NBA/Training Model/nba_reb_ridge_model.pkl

--- Training REGRESSION Model for: AST ---
Best Ridge alpha for AST: 100.0
  - Final Model MAE for AST: 1.50
✅ Saved AST model to: /content/drive/MyDrive/NBA/Training Model/nba_ast_ridge_model.pkl

--- Training REGRES

**Script 3:** The Definitive Prediction & Multi-Sheet Output Engine (FINALIZED)

In [ ]:
# ==============================================================================
# SCRIPT 3: The Definitive Prediction & Multi-Sheet Output Engine (FINALIZED)
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install selenium xlsxwriter nba_api scikit-learn==1.3.0 pandas joblib numpy==1.26.4 pytz unidecode requests
!apt-get update --quiet
!apt-get install -y chromium-chromedriver --quiet

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import requests
import re
import time
import os
import joblib
import pytz
import glob
import json
from datetime import datetime, timezone, timedelta
from unidecode import unidecode
from io import StringIO
from google.colab import drive
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashplayerstats, teamestimatedmetrics, playergamelog, drafthistory
from IPython.display import display
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from xlsxwriter.utility import xl_col_to_name

# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
API_TIMEOUT = 120; API_RETRY_ATTEMPTS = 3; API_RETRY_DELAY = 3; INTER_PLAYER_DELAY = 0.6
ODDS_API_KEYS = [
"9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1",
"146dff42ae19a83208b059d4855095c9", "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9",
"9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b", "66953d96fcb2ec72a47e7f39901258e7",
"94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
"9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f",
"bd6c88c66ff4c7c057aafaffcdd1b8d", "1bfcac154fb2b490bf55a80a18275db1"
]
CONFIG = { "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15, "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20 }

# ==============================================================================
# Part 2: Helper Functions and Data Fetching
# ==============================================================================
def normalize_player_name(name):
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name
def get_implied_prob_from_decimal(odd):
    if odd is None or pd.isna(odd) or float(odd) < 1: return None
    return 1 / float(odd)
def decimal_to_american(decimal_odds):
    if pd.isna(decimal_odds) or decimal_odds is None: return None
    if decimal_odds >= 2.00: return f"+{int((decimal_odds - 1) * 100)}"
    else: return f"-{int(100 / (decimal_odds - 1))}"
def get_current_nba_season():
    now = datetime.now()
    if now.month >= 8: start_year, end_year = now.year, now.year + 1
    else: end_year, start_year = now.year, now.year - 1
    return f"{start_year}-{str(end_year)[-2:]}"
def get_draft_class_ids(season_year):
    print(f"--- Fetching {season_year} Draft Class ---")
    try:
        draft_history_data = drafthistory.DraftHistory(season_year_nullable=season_year)
        df_draft = draft_history_data.get_data_frames()[0]; return set(df_draft['PERSON_ID'].unique())
    except Exception as e: print(f"Could not fetch draft class for {season_year}: {e}"); return set()
def identify_team_changes(master_df):
    print("--- Identifying Players on New Teams ---")
    changed_teams_df = master_df[(master_df['TEAM_ABBREVIATION'] != master_df['CURRENT_TEAM']) & (master_df['CURRENT_TEAM'].notna()) & (master_df['TEAM_ABBREVIATION'].notna())]
    return set(changed_teams_df['PLAYER_ID'].unique())
def get_player_experience(row, rookie_ids, traded_player_ids):
    player_id, games_played_last_season, games_played_this_season = row['PLAYER_ID'], row.get('GP', 99), row.get('GP_current', 0)
    is_rookie = player_id in rookie_ids; is_traded = player_id in traded_player_ids
    is_acclimated = games_played_this_season >= CONFIG["GAMES_PLAYED_THRESHOLD_ACCLIMATED"]
    if is_rookie: return "Traded Rookie" if is_traded and not is_acclimated else "Rookie"
    else:
        if is_traded and not is_acclimated: return "Traded Veteran"
        if games_played_last_season < CONFIG["GAMES_PLAYED_THRESHOLD_LOW_VOLUME"]: return "Low Volume"
        return "Veteran"
def scrape_bref_positions_by_team(season_end_year, teams_to_scrape):
    print(f"--- Scraping B-Ref rosters for {len(teams_to_scrape)} teams for season {season_end_year} ---")
    all_players_df_list = []; driver = None; teams_to_process = list(teams_to_scrape)
    try:
        chrome_options = Options(); chrome_options.add_argument('--headless'); chrome_options.add_argument('--no-sandbox'); chrome_options.add_argument('--disable-dev-shm-usage')
        driver = webdriver.Chrome(options=chrome_options)
        while teams_to_process:
            team_abbr_bref = teams_to_process.pop(0)
            try:
                print(f"Scraping {team_abbr_bref}...")
                url = f"https://www.basketball-reference.com/teams/{team_abbr_bref}/{season_end_year}.html"
                driver.get(url); WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.ID, "roster")))
                tables = pd.read_html(StringIO(driver.page_source), attrs={'id': 'roster'})
                if tables: team_df = tables[0]; team_df['CURRENT_TEAM_BRef'] = team_abbr_bref; all_players_df_list.append(team_df)
            except Exception as e: print(f"  - Error scraping {team_abbr_bref}: {e}"); teams_to_process.append(team_abbr_bref)
    finally:
        if driver: driver.quit()
    if not all_players_df_list: print("CRITICAL ERROR: Failed to scrape any team data."); return pd.DataFrame()
    all_players_df = pd.concat(all_players_df_list, ignore_index=True)
    all_players_df.rename(columns={'Player': 'BRef_Name', 'Pos': 'DetailedPos'}, inplace=True)
    return all_players_df[['BRef_Name', 'DetailedPos', 'CURRENT_TEAM_BRef']]
def fetch_nba_api_player_stats(season):
    print(f"Fetching player stats from nba_api for season: {season}...")
    try: return leaguedashplayerstats.LeagueDashPlayerStats(season=season, per_mode_detailed="PerGame").get_data_frames()[0]
    except Exception as e: print(f"Error: {e}"); return pd.DataFrame()
def fetch_team_metrics(season):
    print(f"Fetching team metrics for season: {season}...")
    try:
        metrics = teamestimatedmetrics.TeamEstimatedMetrics(season=season).get_data_frames()[0]
        team_map = {team['id']: team['abbreviation'] for team in teams.get_teams()}; metrics['TEAM_ABBREVIATION'] = metrics['TEAM_ID'].map(team_map)
        return metrics
    except Exception as e: print(f"Error: {e}"); return pd.DataFrame()
feature_cache = {};
def get_live_features_for_player(player_id, stats_season):
    if player_id in feature_cache: return feature_cache[player_id]
    print(f"  - Engineering features for player ID: {player_id}...")
    logs = pd.DataFrame()
    for attempt in range(API_RETRY_ATTEMPTS):
        try:
            logs = playergamelog.PlayerGameLog(player_id=player_id, season=stats_season, timeout=API_TIMEOUT).get_data_frames()[0]
            if not logs.empty: break
        except Exception as e: print(f"    - Attempt {attempt + 1} failed. Retrying..."); time.sleep(API_RETRY_DELAY)
    if logs.empty or len(logs) < 10: return None
    logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE']); logs = logs.sort_values('GAME_DATE', ascending=True)
    features = {}; stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']; windows = [3, 5, 10]
    for stat in stats_to_roll:
        for w in windows: features[f'ROLL_AVG_{stat}_{w}'] = logs[stat].tail(w).mean()
    days_since_last_game = (datetime.now() - logs.iloc[-1]['GAME_DATE']).days
    features['DAYS_OF_REST'] = min(days_since_last_game, 7)
    feature_cache[player_id] = features
    return features

# ==============================================================================
# Part 4: The Main Prediction Engine
# ==============================================================================
def run_daily_projections():
    try:
        drive.mount('/content/drive', force_remount=True)
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"
        OUTPUT_DIR = "/content/drive/MyDrive/NBA/Predictions"
        os.makedirs(MODEL_DIR, exist_ok=True)
        os.makedirs(OUTPUT_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    print("\n--- Loading Trained Models & Performance Data ---")
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']; classification_props = ['DD', 'TD']
    models = {}; mae_values = {}
    try:
        for prop in regression_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_ridge_model.pkl"))
        for prop in classification_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_logreg_model.pkl"))
        with open(os.path.join(MODEL_DIR, "mae_values.json"), 'r') as f: mae_values = json.load(f)
        print(f"Successfully loaded {len(models)} models and {len(mae_values)} MAE values.")
    except FileNotFoundError as e: print(f"CRITICAL ERROR: Could not load a model or MAE file. Please run Script 2. Missing file: {e}"); return

    print("\n--- Step 1: Fetching today's game schedule from Odds API ---")
    eastern = pytz.timezone('US/Eastern'); now_et = datetime.now(eastern)
    start_of_day_et = now_et.replace(hour=8, minute=0, second=0, microsecond=0); end_of_day_et = start_of_day_et + timedelta(days=1)
    start_utc_str = start_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ'); end_utc_str = end_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    working_key, events = None, []
    for i, api_key in enumerate(ODDS_API_KEYS):
        try:
            response = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/events', params={'apiKey': api_key, 'commenceTimeFrom': start_utc_str, 'commenceTimeTo': end_utc_str}, timeout=30)
            response.raise_for_status(); events = response.json(); working_key = api_key; break
        except Exception as e: print(f"--- API Key #{i+1} failed. Error: {e} ---")
    if not working_key: print("--- CRITICAL: All API keys failed. ---"); return
    if not events: print("No NBA games scheduled for today."); return
    team_name_map = {team['full_name']: team['abbreviation'] for team in teams.get_teams()}
    games_today = [{'home_team_abbr': team_name_map.get(e['home_team']), 'away_team_abbr': team_name_map.get(e['away_team'])} for e in events if team_name_map.get(e['home_team']) and team_name_map.get(e['away_team'])]
    games_today = [dict(t) for t in {tuple(d.items()) for d in games_today}]

    print("\n--- Step 2: Acquiring Player Rosters ---"); today_str = now_et.strftime('%Y-%m-%d')
    CURRENT_SEASON = get_current_nba_season(); current_year_start = int(CURRENT_SEASON.split('-')[0])
    POSITIONS_SEASON_END_YEAR = current_year_start + 1
    cache_dir = "/content/drive/MyDrive/NBA"; cache_filename = f"bref_roster_cache_{today_str}.csv"; cache_path = os.path.join(cache_dir, cache_filename)
    for old_cache_file in glob.glob(os.path.join(cache_dir, "bref_roster_cache_*.csv")):
        if old_cache_file != cache_path:
            try: os.remove(old_cache_file)
            except OSError as e: print(f"-> Error removing old cache file {os.path.basename(old_cache_file)}: {e}")
    cached_bref_df = pd.DataFrame()
    if os.path.exists(cache_path):
        try: cached_bref_df = pd.read_csv(cache_path)
        except Exception as e: print(f"-> Warning: Could not read today's cache file. Error: {e}")
    teams_playing_today = {g['home_team_abbr'] for g in games_today} | {g['away_team_abbr'] for g in games_today}
    teams_needed = list(teams_playing_today - set(cached_bref_df['CURRENT_TEAM'].unique())) if not cached_bref_df.empty else list(teams_playing_today)
    if teams_needed:
        bref_team_map = {"BKN": "BRK", "PHX": "PHO", "CHA": "CHO"}; teams_to_scrape_bref = [bref_team_map.get(team, team) for team in teams_needed]
        scraped_data = scrape_bref_positions_by_team(season_end_year=POSITIONS_SEASON_END_YEAR, teams_to_scrape=teams_to_scrape_bref)
        if not scraped_data.empty:
            reverse_bref_map = {v: k for k, v in bref_team_map.items()}
            scraped_data['CURRENT_TEAM'] = scraped_data['CURRENT_TEAM_BRef'].apply(lambda x: reverse_bref_map.get(x, x)); scraped_data.drop(columns=['CURRENT_TEAM_BRef'], inplace=True)
            bref_df = pd.concat([cached_bref_df, scraped_data], ignore_index=True).drop_duplicates(subset=['BRef_Name'], keep='last')
        else: bref_df = cached_bref_df
    else: bref_df = cached_bref_df
    scraped_teams = set(bref_df['CURRENT_TEAM'].unique()) if not bref_df.empty else set()
    if not teams_playing_today.issubset(scraped_teams):
        print("\nCRITICAL FAILURE: Could not scrape rosters for all teams playing today.");
        if not bref_df.empty: bref_df.to_csv(cache_path, index=False)
        return
    else:
        if not bref_df.empty: bref_df.to_csv(cache_path, index=False)

    print("\n--- Step 3: Fetching player prop odds from Odds API ---")
    MARKETS_TO_KEEP = {'player_points': 'PTS', 'player_rebounds': 'REB', 'player_assists': 'AST', 'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A', 'player_double_double': 'DD', 'player_triple_double': 'TD'}
    all_props = []; current_key_index = ODDS_API_KEYS.index(working_key)
    for event in events:
        event_id = event['id']; print(f"  - Fetching props for game: {event['away_team']} @ {event['home_team']}...")
        odds_fetched = False
        while not odds_fetched and current_key_index < len(ODDS_API_KEYS):
            api_key = ODDS_API_KEYS[current_key_index]; params = {'apiKey': api_key, 'regions': 'us', 'markets': ','.join(MARKETS_TO_KEEP.keys()), 'bookmakers': 'draftkings,fanduel'}
            try:
                response = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds', params=params, timeout=30); response.raise_for_status()
                for bookmaker in response.json().get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        prop_type = MARKETS_TO_KEEP.get(market['key'])
                        if prop_type:
                            for outcome in market.get('outcomes', []): all_props.append({'Game': f"{team_name_map.get(event['away_team'])} @ {team_name_map.get(event['home_team'])}", 'Game_Time': event['commence_time'], 'Player': outcome['description'], 'Prop': prop_type, 'Line': outcome.get('point'), 'Over_Odds': outcome['price'] if outcome['name'] == 'Over' else None, 'Under_Odds': outcome['price'] if outcome['name'] == 'Under' else None, 'Yes_Odds': outcome['price'] if outcome['name'] == 'Yes' else None, 'No_Odds': outcome['price'] if outcome['name'] == 'No' else None, 'Bookmaker': bookmaker['title']})
                odds_fetched = True
            except Exception as e: print(f"    - Key #{current_key_index + 1} failed. Error: {e}"); current_key_index += 1
    if not all_props: odds_df = pd.DataFrame(); print("No player props were available. Aborting."); return
    df = pd.DataFrame(all_props); df['match_key'] = df['Player'].apply(normalize_player_name)
    odds_df_ou = df[df['Prop'].isin(regression_props)].groupby(['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time']).agg({'Over_Odds': 'first', 'Under_Odds': 'first'}).reset_index()
    odds_df_yn = df[df['Prop'].isin(classification_props)].groupby(['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time']).agg({'Yes_Odds': 'first', 'No_Odds': 'first'}).reset_index()
    odds_df_yn['Implied_Prob_Yes'] = odds_df_yn['Yes_Odds'].apply(get_implied_prob_from_decimal)
    odds_df = pd.concat([odds_df_ou, odds_df_yn], ignore_index=True)
    for col in ['Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds']: odds_df[col] = odds_df[col].apply(decimal_to_american)
    odds_df['Game_Time'] = pd.to_datetime(odds_df['Game_Time']).dt.tz_convert('US/Eastern').dt.strftime('%-I:%M %p EST')

    print("\n--- Step 4: Preparing Data for Projections ---")
    use_previous_season_stats = today_str < f"{current_year_start}-12-01"
    STATS_SEASON_STRING = f"{current_year_start - 1}-{str(current_year_start)[-2:]}" if use_previous_season_stats else CURRENT_SEASON
    CURRENT_DRAFT_YEAR_STR = str(current_year_start)
    nba_df = fetch_nba_api_player_stats(season=STATS_SEASON_STRING)
    nba_df_current = fetch_nba_api_player_stats(season=CURRENT_SEASON) if use_previous_season_stats else nba_df
    team_metrics_df = fetch_team_metrics(season=STATS_SEASON_STRING); rookie_ids = get_draft_class_ids(CURRENT_DRAFT_YEAR_STR)
    if nba_df.empty or team_metrics_df.empty: print("\nAborting: missing critical data."); return

    nba_df['merge_key'] = nba_df['PLAYER_NAME'].apply(normalize_player_name); bref_df['merge_key'] = bref_df['BRef_Name'].apply(normalize_player_name)
    master_df = pd.merge(nba_df, bref_df, on='merge_key', how='inner')
    if master_df.empty: print("\nWARNING: No players matched."); return
    if use_previous_season_stats and not nba_df_current.empty: master_df['GP_current'] = master_df['PLAYER_ID'].map(nba_df_current.set_index('PLAYER_ID')['GP']).fillna(0)
    else: master_df['GP_current'] = master_df['GP']
    traded_player_ids = identify_team_changes(master_df)

    players_with_odds = odds_df['match_key'].unique(); players_to_project_df = master_df[master_df['merge_key'].isin(players_with_odds)]
    print(f"\n--- Generating Projections for {len(players_to_project_df)} Players with Active Props ---")

    all_projections = []
    for _, player in players_to_project_df.iterrows():
        time.sleep(INTER_PLAYER_DELAY); player_features = get_live_features_for_player(player['PLAYER_ID'], STATS_SEASON_STRING)
        if not player_features: continue
        player_features['GP'] = player['GP']
        opp_team_abbr = next((g['home_team_abbr'] if player['CURRENT_TEAM'] == g['away_team_abbr'] else g['away_team_abbr'] for g in games_today if player['CURRENT_TEAM'] in [g['home_team_abbr'], g['away_team_abbr']]), None)
        if not opp_team_abbr or team_metrics_df[team_metrics_df['TEAM_ABBREVIATION'] == opp_team_abbr].empty: continue
        opp_metrics = team_metrics_df[team_metrics_df['TEAM_ABBREVIATION'] == opp_team_abbr].iloc[0]
        player_features['E_PACE_OPP'] = opp_metrics['E_PACE']; player_features['E_DEF_RATING_OPP'] = opp_metrics['E_DEF_RATING']
        projections_for_player = {'Player': player['PLAYER_NAME'], 'Team': player['CURRENT_TEAM'], 'Pos': player['DetailedPos'], 'match_key': player['merge_key'], 'Experience': get_player_experience(player, rookie_ids, traded_player_ids)}
        feature_df = pd.DataFrame([player_features])[models['PTS'].named_steps['scaler'].get_feature_names_out()]
        for prop in regression_props: projections_for_player[f'Proj_{prop}'] = round(max(0, models[prop].predict(feature_df)[0]), 2)
        for prop in classification_props: projections_for_player[f'Proj_{prop}_%'] = f"{models[prop].predict_proba(feature_df)[0][1]:.1%}"
        all_projections.append(projections_for_player)

    if not all_projections: print("\n--- WARNING: No projections were generated. ---"); return
    projections_df = pd.DataFrame(all_projections)

    print("\n--- Step 5: Calculating Edge and Generating Output Files ---")
    melted_projections = projections_df.melt(id_vars=['match_key', 'Player', 'Team', 'Pos', 'Experience'], value_vars=[c for c in projections_df.columns if c.startswith('Proj')], var_name='Prop_Name', value_name='Projection')
    melted_projections['Prop'] = melted_projections['Prop_Name'].str.replace('Proj_', '').str.replace('_%', '')
    master_long_df = pd.merge(melted_projections, odds_df, on=['match_key', 'Prop'], how='left', suffixes=('_proj', '_odds'))
    master_long_df.dropna(subset=['Game'], inplace=True)
    master_long_df['Player'] = master_long_df['Player_proj'].fillna(master_long_df['Player_odds']); master_long_df.drop(columns=['Player_proj', 'Player_odds'], inplace=True, errors='ignore')
    master_long_df['Projection_Num'] = pd.to_numeric(master_long_df['Projection'].astype(str).str.replace('%', ''), errors='coerce')
    master_long_df.drop(columns=['Projection'], inplace=True, errors='ignore')

    yn_mask = master_long_df['Prop'].isin(classification_props)
    master_long_df['DD & TD\nModel Score'] = np.nan
    master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] = master_long_df.loc[yn_mask, 'Projection_Num']
    master_long_df.loc[yn_mask, 'Projection_Num'] = np.nan

    def create_range_string(row):
        if pd.notna(row['Proj Range (Low)']): return f"{row['Proj Range (Low)']}-{row['Proj Range (High)']}"
        return ""
    master_long_df['Proj Range (Low)'] = np.nan; master_long_df['Proj Range (High)'] = np.nan
    for prop, mae in mae_values.items():
        mask = master_long_df['Prop'] == prop
        master_long_df.loc[mask, 'Proj Range (Low)'] = (master_long_df.loc[mask, 'Projection_Num'] - mae).round(1).apply(lambda x: max(0, x))
        master_long_df.loc[mask, 'Proj Range (High)'] = (master_long_df.loc[mask, 'Projection_Num'] + mae).round(1)
    master_long_df['Proj Range'] = master_long_df.apply(create_range_string, axis=1)
    master_long_df.drop(columns=['Proj Range (Low)', 'Proj Range (High)'], inplace=True)

    master_long_df['Edge'] = None; master_long_df['Bet'] = 'Pass'
    ou_mask = master_long_df['Prop'].isin(regression_props)
    master_long_df.loc[ou_mask, 'Edge'] = ((master_long_df.loc[ou_mask, 'Projection_Num'] - master_long_df.loc[ou_mask, 'Line']) / master_long_df.loc[ou_mask, 'Line']) * 100
    master_long_df.loc[ou_mask, 'Bet'] = master_long_df.loc[ou_mask, 'Edge'].apply(lambda x: 'Over' if pd.notna(x) and x > 10 else ('Under' if pd.notna(x) and x < -10 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns: master_long_df.loc[yn_mask, 'Edge'] = master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] - (master_long_df.loc[yn_mask, 'Implied_Prob_Yes'].fillna(0) * 100)
    else: master_long_df.loc[yn_mask, 'Edge'] = None
    master_long_df.loc[yn_mask, 'Bet'] = master_long_df.loc[yn_mask, 'Edge'].apply(lambda x: 'Yes' if pd.notna(x) and x > 5 else ('No' if pd.notna(x) and x < -5 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns: master_long_df.drop(columns=['Implied_Prob_Yes'], inplace=True)

    output_filename = f"{now_et.strftime('%m-%d-%Y')} NBA Props Predictions.xlsx"; full_output_path = os.path.join(OUTPUT_DIR, output_filename)
    with pd.ExcelWriter(full_output_path, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'fg_color': '#D7E4BC', 'border': 1, 'text_wrap': True})
        percent_format = workbook.add_format({'num_format': '0.0"%"', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        center_format = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        exceptional_format = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        strong_format = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C6500', 'align': 'center', 'valign': 'vcenter', 'border': 1})

        all_ev_bets_df = master_long_df[master_long_df['Bet'] != 'Pass'].copy()
        over_under_bets = all_ev_bets_df[all_ev_bets_df['Bet'].isin(['Over', 'Under'])]
        yes_dd_td_bets = all_ev_bets_df[(all_ev_bets_df['Bet'] == 'Yes') & (all_ev_bets_df['Prop'].isin(['DD', 'TD']))]
        ev_bets_df = pd.concat([over_under_bets, yes_dd_td_bets])
        if not ev_bets_df.empty:
            for bookmaker in ['DraftKings', 'FanDuel']:
                book_df = ev_bets_df[ev_bets_df['Bookmaker'] == bookmaker]
                if book_df.empty: continue
                dfs_by_game = [game_bets.sort_values(by='Edge', key=abs, ascending=False) for game_tuple in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])) if not (game_bets := book_df[book_df['Game'] == f"{game_tuple['away_team_abbr']} @ {game_tuple['home_team_abbr']}"]).empty]
                if dfs_by_game:
                    data_only_df = pd.concat(dfs_by_game, ignore_index=True)
                    final_dfs_for_excel = [];
                    for _, game_group in data_only_df.groupby('Game', sort=False):
                        final_dfs_for_excel.append(game_group)
                        final_dfs_for_excel.append(pd.DataFrame([pd.NA] * len(game_group.columns), index=game_group.columns).T)
                    final_dfs_for_excel.pop()
                    final_ev_df = pd.concat(final_dfs_for_excel, ignore_index=True)
                    final_ev_df.rename(columns={'Projection_Num': 'Projection'}, inplace=True)

                    ev_cols = ['Game_Time', 'Game', 'Player', 'Team', 'Prop', 'Line', 'DD & TD\nModel Score', 'Projection', 'Proj Range', 'Edge', 'Bet', 'Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds']
                    final_ev_df = final_ev_df.reindex(columns=ev_cols); sheet_name = f"{bookmaker}_EV_Bets"

                    final_ev_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
                    worksheet = writer.sheets[sheet_name]
                    for col_num, value in enumerate(final_ev_df.columns.values):
                        worksheet.write(0, col_num, value, header_format)
                        data_len = final_ev_df[value].astype(str).map(len).max() if not final_ev_df[value].isnull().all() else 0
                        header_len = max([len(s) for s in value.split('\n')])
                        column_width = max(data_len, header_len) + 2
                        col_format = percent_format if value == 'Edge' else center_format
                        worksheet.set_column(col_num, col_num, column_width, col_format)
                    worksheet.set_row(0, 30); worksheet.freeze_panes(1, 0)

                    data_range = f'A2:{xl_col_to_name(len(ev_cols)-1)}{len(final_ev_df)+1}'
                    # --- [FINAL BUG FIX] Use correct newline character \n ---
                    prop_col = f'${xl_col_to_name(ev_cols.index("Prop"))}'; edge_col = f'${xl_col_to_name(ev_cols.index("Edge"))}'; score_col = f'${xl_col_to_name(ev_cols.index("DD & TD\nModel Score"))}'
                    worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND({prop_col}2<>"DD", {prop_col}2<>"TD", ABS({edge_col}2)>=25)', 'format': exceptional_format})
                    worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND(OR({prop_col}2="DD", {prop_col}2="TD"), {score_col}2>=75, {edge_col}2>=40)', 'format': exceptional_format})
                    worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND({prop_col}2<>"DD", {prop_col}2<>"TD", ABS({edge_col}2)>=15)', 'format': strong_format})
                    worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND(OR({prop_col}2="DD", {prop_col}2="TD"), {score_col}2>=65, {edge_col}2>=20)', 'format': strong_format})
                    print(f"  - Generated sheet: {sheet_name}")

        if not projections_df.empty:
            for game in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
                home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
                sheet_name = f"{away_abbr}_vs_{home_abbr}_Full_Data"
                game_df = master_long_df[master_long_df['Game'] == f"{away_abbr} @ {home_abbr}"].copy()
                if game_df.empty: continue
                game_df.rename(columns={'Projection_Num': 'Projection'}, inplace=True)
                final_cols = ['Player', 'Team', 'Prop', 'Line', 'DD & TD\nModel Score', 'Projection', 'Proj Range', 'Edge', 'Bet', 'Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds', 'Bookmaker']
                game_long_df = game_df.reindex(columns=final_cols).sort_values(by=['Player', 'Prop'])

                game_long_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
                worksheet = writer.sheets[sheet_name]
                for col_num, value in enumerate(game_long_df.columns.values):
                    worksheet.write(0, col_num, value, header_format)
                    data_len = game_long_df[value].astype(str).map(len).max() if not game_long_df[value].isnull().all() else 0
                    header_len = max([len(s) for s in value.split('\n')])
                    column_width = max(data_len, header_len) + 2
                    col_format = percent_format if value == 'Edge' else center_format
                    worksheet.set_column(col_num, col_num, column_width, col_format)
                worksheet.set_row(0, 30); worksheet.freeze_panes(1, 0)

                data_range = f'A2:{xl_col_to_name(len(final_cols)-1)}{len(game_long_df)+1}'
                # --- [FINAL BUG FIX] Use correct newline character \n ---
                prop_col = f'${xl_col_to_name(final_cols.index("Prop"))}'; edge_col = f'${xl_col_to_name(final_cols.index("Edge"))}'; score_col = f'${xl_col_to_name(final_cols.index("DD & TD\nModel Score"))}'
                worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND({prop_col}2<>"DD", {prop_col}2<>"TD", ABS({edge_col}2)>=25)', 'format': exceptional_format})
                worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND(OR({prop_col}2="DD", {prop_col}2="TD"), {score_col}2>=75, {edge_col}2>=40)', 'format': exceptional_format})
                worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND({prop_col}2<>"DD", {prop_col}2<>"TD", ABS({edge_col}2)>=15)', 'format': strong_format})
                worksheet.conditional_format(data_range, {'type': 'formula', 'criteria': f'=AND(OR({prop_col}2="DD", {prop_col}2="TD"), {score_col}2>=65, {edge_col}2>=20)', 'format': strong_format})
                print(f"  - Generated sheet: {sheet_name}")
    print(f"\n✅ Successfully exported all data to: {full_output_path}")

if __name__ == "__main__":
    pd.set_option('display.max_rows', 200); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000)
    run_daily_projections()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 27.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 28.2 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 htt

ModuleNotFoundError: No module named 'unidecode'